# Autoencoders y Variational Autoencoders

**Curso:** Deep Learning — Posgrado en Ciencia de Datos  
**Duración sugerida:** 5 horas (2 h Autoencoders + 3 h Variational Autoencoders)  
**Framework:** TensorFlow 2.x / Keras

---

## Mapa de la sesión

| Bloque | Contenido | Resultado matemático | Tiempo |
|---|---|---|---|
| **I.1** | Autoencoders: definición y entrenamiento | — | 20 min |
| **I.2** | AE lineal y PCA | **Teorema 1** (Baldi–Hornik) | 25 min |
| **I.3** | AE profundos en TensorFlow | — | 25 min |
| **I.4** | AE regularizados: denoising, sparse, contractivo | **Teorema 2** (Alain–Bengio) | 35 min |
| **I.5** | Límites del AE como modelo generativo | motivación | 15 min |
| **II.1** | Modelos de variable latente e inferencia variacional | Definiciones | 25 min |
| **II.2** | ELBO | **Teorema 3** (descomposición de la evidencia) | 35 min |
| **II.3** | Pérdida del VAE | **Teorema 4** (KL gaussiana cerrada) | 20 min |
| **II.4** | Truco de reparametrización | **Teorema 5** | 30 min |
| **II.5** | VAE en TensorFlow (MNIST) | — | 35 min |
| **II.6** | Cotas más ajustadas: IWAE | **Teorema 6** (monotonía) | 15 min |
| **II.7** | Tasa–distorsión y $\beta$-VAE | **Teorema 7** | 15 min |
| **II.8** | Colapso posterior, CVAE y ejercicios | — | 20 min |

> **Objetivos de aprendizaje.** Al terminar, el estudiante podrá: (i) demostrar que un AE lineal recupera el subespacio PCA; (ii) explicar qué estima un AE regularizado sobre la densidad de los datos; (iii) derivar el ELBO y su descomposición; (iv) justificar por qué la reparametrización reduce la varianza del gradiente; (v) implementar y diagnosticar un VAE en TensorFlow.

**Convención.** Los teoremas incluyen demostración y una **verificación numérica**. Las referencias están numeradas $[n]$ y se listan al final.

In [1]:
# ==== Configuración global (ejecutar primero) ====
import os, math, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25})
print("TensorFlow:", tf.__version__)
print("GPU disponible:", bool(tf.config.list_physical_devices("GPU")))

TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

---
# PARTE I — Autoencoders
---
## I.1 Definición y entrenamiento

````{figure} ./imgs/autoencoder-architecture.png
:align: center

````

- Los ***autoencoders*** han sido propuestos como ***métodos para la reducción de dimensionalidad***. Un ***autoencoder*** consta de dos partes, el ***codificador*** y el ***decodificador***. La ***salida del `codificador es la representación reducida del patrón de entrada`***, y se define en términos de una función vectorial

    $$
    \boldsymbol{g}_{\phi}:~\boldsymbol{x}\in\mathbb{R}^{l}\longmapsto\boldsymbol{z}\in\mathbb{R}^{m},
    $$

    donde 
    
    $$
    z_{i}:=g_{\phi, i}(\boldsymbol{x})=\phi_{e}(\boldsymbol{\theta}_{i}^{T}\boldsymbol{x}+b_{i}),\quad i=1,2,\dots,m,
    $$

    con $\phi_{e}$ función de ***activación***; esta última suele tomarse como la función ***sigmoidea logística*** $\phi_{e}(\cdot)=\sigma(\cdot)$. En otras palabras, el ***codificador es una red neuronal feed-forward de una sola capa oculta***

- El ***`decodificador`*** es otra función $\boldsymbol{f}_{\theta}$.

    $$
    \boldsymbol{f}_{\theta}:~\boldsymbol{z}\in\mathbb{R}^{m}\longmapsto\hat{\boldsymbol{x}}\in\mathbb{R}^{l},
    $$

    donde 

    $$
    \hat{x}_{j}=f_{\theta, j}(\boldsymbol{z})=\phi_{d}(\boldsymbol{\theta}_{j}'^{T}\boldsymbol{z}+b_{j}'),\quad j=1,\dots,l.
    $$

    La función de ***activación*** $\phi_{d}$ suele ser la identidad (***reconstrucción lineal***) o la ***sigmoidea logística***.

- La tarea de entrenamiento consiste en ***estimar los parámetros***

$$
\Theta:=[\boldsymbol{\theta}_{1},\dots,\boldsymbol{\theta}_{m}],~\boldsymbol{b},~\Theta':=[\boldsymbol{\theta}_{1}',\dots,\boldsymbol{\theta}_{l}'],~\boldsymbol{b}'.
$$

- Es habitual suponer que $\Theta'=\Theta^{T}$ (***pesos atados***, *tied weights*). Los parámetros se estiman para que el ***error de reconstrucción*** $\boldsymbol{e}=\boldsymbol{x}-\hat{\boldsymbol{x}}$, sobre las muestras de entrada disponibles sea ***mínimo***. Normalmente, se utiliza el ***coste por mínimos cuadrados*** pero también son posibles otras opciones.
- Las ***versiones regularizadas***, que implican una norma de los parámetros, también es una posibilidad. Si se elige que la activación $\phi_{e}$ sea la identidad (representación lineal) y $m < l$ (para evitar la trivialidad), el ***autoencoder es equivalente a la técnica PCA***.

> La última afirmación es exactamente el contenido del **Teorema 1**, que enunciamos y demostramos a continuación.

### Formulación como problema de optimización

**Definición 1 (Autoencoder).** Sea $\mathcal{X}\subseteq\mathbb{R}^{l}$ el espacio de datos y $m<l$ la dimensión del código. Un *autoencoder* es un par $(g_\phi, f_\theta)$ con $g_\phi:\mathbb{R}^l\to\mathbb{R}^m$ (codificador) y $f_\theta:\mathbb{R}^m\to\mathbb{R}^l$ (decodificador), entrenado para resolver

$$
\min_{\phi,\theta}\;\; \mathcal{J}(\phi,\theta)=\frac{1}{N}\sum_{n=1}^{N}\ell\big(\boldsymbol{x}^{(n)},\, f_\theta(g_\phi(\boldsymbol{x}^{(n)}))\big),
\qquad \ell(\boldsymbol{x},\hat{\boldsymbol{x}})=\|\boldsymbol{x}-\hat{\boldsymbol{x}}\|_2^2 .
$$

**Observación (cuello de botella).** Si $m\ge l$ y no hay restricción adicional, la identidad es solución y el AE no aprende nada útil. La restricción $m<l$ (*undercomplete*) o una **regularización** fuerzan al modelo a capturar estructura de los datos.

**Interpretación probabilística.** Minimizar el error cuadrático equivale a maximizar la log-verosimilitud de un decodificador gaussiano con varianza fija: $p_\theta(\boldsymbol{x}|\boldsymbol{z})=\mathcal{N}(\boldsymbol{x};f_\theta(\boldsymbol{z}),\sigma^2 I)$, pues $-\log p_\theta(\boldsymbol{x}|\boldsymbol{z})=\tfrac{1}{2\sigma^2}\|\boldsymbol{x}-f_\theta(\boldsymbol{z})\|^2+\text{cte}$. Esta lectura será clave en la Parte II.

---
## I.2 Autoencoder lineal y PCA

Recordemos el resultado clásico de PCA. Sea $\boldsymbol{X}\in\mathbb{R}^{N\times l}$ con filas centradas y covarianza muestral $\Sigma=\tfrac1N\boldsymbol{X}^T\boldsymbol{X}$, con autovalores $\lambda_1\ge\dots\ge\lambda_l\ge0$ y autovectores ortonormales $\boldsymbol{v}_1,\dots,\boldsymbol{v}_l$.

**Teorema (Eckart–Young–Mirsky / PCA).** Entre todas las matrices $\boldsymbol{P}$ de proyección ortogonal de rango $m$, el error $\mathbb{E}\|\boldsymbol{x}-\boldsymbol{P}\boldsymbol{x}\|^2$ se minimiza con $\boldsymbol{P}=\boldsymbol{V}_m\boldsymbol{V}_m^T$, $\boldsymbol{V}_m=[\boldsymbol{v}_1,\dots,\boldsymbol{v}_m]$, y vale $\sum_{i>m}\lambda_i$ [5].

Nos preguntamos: ¿qué aprende un autoencoder **lineal** entrenado con mínimos cuadrados?

### Teorema 1 (Baldi–Hornik, 1989)

> **Teorema 1.** Sea $\Sigma$ la matriz de covarianza de los datos centrados, **de rango completo con autovalores distintos** $\lambda_1>\dots>\lambda_l>0$. Considere el autoencoder lineal $\hat{\boldsymbol{x}}=\boldsymbol{W}_2\boldsymbol{W}_1\boldsymbol{x}$, con $\boldsymbol{W}_1\in\mathbb{R}^{m\times l}$, $\boldsymbol{W}_2\in\mathbb{R}^{l\times m}$, $m<l$, y la pérdida
> $$\mathcal{J}(\boldsymbol{W}_1,\boldsymbol{W}_2)=\mathbb{E}\,\|\boldsymbol{x}-\boldsymbol{W}_2\boldsymbol{W}_1\boldsymbol{x}\|^2 .$$
> Entonces:
> 1. **(Puntos críticos)** $(\boldsymbol{W}_1,\boldsymbol{W}_2)$ es punto crítico si y solo si $\boldsymbol{W}_2\boldsymbol{W}_1=\boldsymbol{P}_{\mathcal{I}}$, la proyección ortogonal sobre el subespacio generado por *algún* conjunto $\mathcal{I}$ de $m$ autovectores de $\Sigma$ (con $\boldsymbol{W}_1$ de rango $m$).
> 2. **(Mínimo global)** El valor mínimo es $\sum_{i>m}\lambda_i$ y se alcanza cuando $\mathcal{I}=\{1,\dots,m\}$, es decir, $\boldsymbol{W}_2\boldsymbol{W}_1=\boldsymbol{V}_m\boldsymbol{V}_m^T$.
> 3. **(Sin mínimos locales)** Todo punto crítico con $\mathcal{I}\neq\{1,\dots,m\}$ es un **punto silla**; no existen mínimos locales que no sean globales.
>
> *Consecuencia:* el AE lineal aprende el **subespacio** principal, **no** los componentes individuales (la solución es única salvo $\boldsymbol{W}_1\mapsto\boldsymbol{A}\boldsymbol{W}_1$, $\boldsymbol{W}_2\mapsto\boldsymbol{W}_2\boldsymbol{A}^{-1}$ con $\boldsymbol{A}\in GL_m$).

**Referencia:** Baldi & Hornik (1989) [1]; ver también Bourlard & Kamp (1988) [2] y Kunin et al. (2019) [3] para la versión con regularización $\ell_2$, que sí desambigua los ejes.

#### Demostración (esquema completo de 1 y 2)

Sea $\boldsymbol{A}=\boldsymbol{W}_2\boldsymbol{W}_1\in\mathbb{R}^{l\times l}$ (rango $\le m$). Expandiendo,

$$
\mathcal{J}=\operatorname{tr}\big[(I-\boldsymbol{A})\Sigma(I-\boldsymbol{A})^T\big].
$$

**Paso 1: gradientes.** Con $\boldsymbol{E}=\boldsymbol{A}-I$,

$$
\nabla_{\boldsymbol{W}_2}\mathcal{J}=2\,\boldsymbol{E}\,\Sigma\,\boldsymbol{W}_1^T,\qquad
\nabla_{\boldsymbol{W}_1}\mathcal{J}=2\,\boldsymbol{W}_2^T\boldsymbol{E}\,\Sigma .
$$

**Paso 2: condiciones de punto crítico.** Si $\boldsymbol{W}_1$ tiene rango $m$ y $\Sigma\succ0$, entonces $\boldsymbol{W}_1\Sigma\boldsymbol{W}_1^T$ es invertible y de $\nabla_{\boldsymbol{W}_2}\mathcal{J}=0$ obtenemos

$$
\boldsymbol{W}_2=\Sigma\boldsymbol{W}_1^T(\boldsymbol{W}_1\Sigma\boldsymbol{W}_1^T)^{-1}.
$$

Sustituyendo en $\nabla_{\boldsymbol{W}_1}\mathcal{J}=0$ se llega a

$$
\boldsymbol{A}\,\Sigma=\Sigma\,\boldsymbol{A}^{T}=\boldsymbol{A}\Sigma\boldsymbol{A}^T .
$$

**Paso 3: $\boldsymbol{A}$ es una proyección ortogonal que conmuta con $\Sigma$.** Sustituyendo $\boldsymbol{W}_2$ en $\nabla_{\boldsymbol{W}_1}\mathcal{J}=0$ se obtiene

$$
\boldsymbol{A}\,\Sigma\,\boldsymbol{A}^{T}=\Sigma\,\boldsymbol{A}^{T}.
$$

El lado izquierdo es **simétrico**, luego también lo es $\Sigma\boldsymbol{A}^T$; es decir $\Sigma\boldsymbol{A}^T=(\Sigma\boldsymbol{A}^T)^T=\boldsymbol{A}\Sigma$. Por tanto

$$
\boldsymbol{A}\Sigma=\Sigma\boldsymbol{A}^T=\boldsymbol{A}\Sigma\boldsymbol{A}^T .
$$

De aquí: (i) $\boldsymbol{A}\Sigma=\Sigma\boldsymbol{A}^T$; (ii) por construcción $\boldsymbol{A}=\boldsymbol{W}_2(\boldsymbol{W}_1\Sigma\boldsymbol{W}_1^T)^{-1}\boldsymbol{W}_1\Sigma$ es **idempotente** ($\boldsymbol{A}^2=\boldsymbol{A}$) y, con (i), simétrica. Luego $\boldsymbol{A}$ es una proyección ortogonal de rango $m$ cuya imagen es un subespacio **invariante** por $\Sigma$. Como los autovalores son distintos, todo subespacio invariante de dimensión $m$ es generado por $m$ autovectores: $\boldsymbol{A}=\boldsymbol{P}_{\mathcal{I}}$.

**Paso 4: valor en el punto crítico.** $\mathcal{J}(\boldsymbol{P}_{\mathcal{I}})=\operatorname{tr}[(I-\boldsymbol{P}_{\mathcal{I}})\Sigma]=\sum_{i\notin\mathcal{I}}\lambda_i$, que es mínimo si y solo si $\mathcal{I}$ contiene los $m$ mayores autovalores. $\qquad\blacksquare$

**Parte 3 (puntos silla).** Si $\mathcal{I}\ne\{1,\dots,m\}$ existe $i\in\mathcal{I}$, $j\notin\mathcal{I}$ con $\lambda_j>\lambda_i$. Perturbando $\boldsymbol{W}_1$ en la dirección que rota $\boldsymbol{v}_i$ hacia $\boldsymbol{v}_j$ el valor de $\mathcal{J}$ **disminuye** a segundo orden; por tanto el hessiano tiene una dirección de curvatura negativa. La demostración detallada está en [1, Sec. 3 y Apéndice].

> **Nota docente.** Pida a los estudiantes que completen el Paso 3 como ejercicio; es un buen ejemplo de cómo la estructura algebraica (conmutación con $\Sigma$) determina por completo el conjunto de soluciones.

#### Verificación numérica del Teorema 1 con TensorFlow

Entrenamos un AE lineal por descenso de gradiente y comparamos (a) su error con $\sum_{i>m}\lambda_i$ y (b) su proyector $\boldsymbol{W}_2\boldsymbol{W}_1$ con $\boldsymbol{V}_m\boldsymbol{V}_m^T$.

In [ ]:
# ---- Datos sintéticos con covarianza conocida ----
rng = np.random.default_rng(0)
N, l, m = 5000, 10, 3
A_mix = rng.normal(size=(l, l))
X_syn = (rng.normal(size=(N, l)) @ A_mix).astype("float32")
X_syn -= X_syn.mean(0)

# ---- PCA de referencia ----
Sigma = X_syn.T @ X_syn / N
lam, V = np.linalg.eigh(Sigma)
order = np.argsort(lam)[::-1]; lam, V = lam[order], V[:, order]
P_pca = V[:, :m] @ V[:, :m].T
print("Error óptimo teórico  sum_{i>m} lambda_i =", lam[m:].sum())

# ---- AE lineal en Keras ----
lin_ae = keras.Sequential([
    keras.Input(shape=(l,)),
    layers.Dense(m, use_bias=False, name="enc"),
    layers.Dense(l, use_bias=False, name="dec"),
])
lin_ae.compile(optimizer=keras.optimizers.Adam(1e-2), loss="mse")
hist = lin_ae.fit(X_syn, X_syn, epochs=300, batch_size=500, verbose=0)

# Keras 'mse' promedia sobre las l coordenadas -> multiplicamos por l para obtener E||x - x_hat||^2
err_ae = hist.history["loss"][-1] * l
W1 = lin_ae.get_layer("enc").get_weights()[0]      # (l, m)
W2 = lin_ae.get_layer("dec").get_weights()[0]      # (m, l)
P_ae = (W1 @ W2).T                                 # W2^T-convención -> operador l x l
print("Error del AE lineal              =", err_ae)
print("||P_AE - P_PCA||_F               =", np.linalg.norm(P_ae - P_pca))

In [ ]:
# Gráfico: el error del AE converge al valor de PCA (cota del Teorema 1)
plt.figure(figsize=(6,3.2))
plt.plot(np.array(hist.history["loss"])*l, label="AE lineal")
plt.axhline(lam[m:].sum(), color="crimson", ls="--", label=r"$\sum_{i>m}\lambda_i$ (PCA)")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel(r"$\mathbb{E}\|x-\hat{x}\|^2$")
plt.title("Teorema 1: el AE lineal alcanza el óptimo PCA"); plt.legend(); plt.show()

In [ ]:
# ---- Teorema 1 (parte 3): los puntos críticos "equivocados" son SILLAS ----
# Subespacio {v1, v2, v5}: es punto crítico, pero rotar v5 hacia v3 (lambda_3 > lambda_5) DISMINUYE J.
rng = np.random.default_rng(1)
l_, m_ = 8, 3
B = rng.normal(size=(l_, l_)); Sg = B @ B.T + np.eye(l_)
lam_, V_ = np.linalg.eigh(Sg); V_ = V_[:, ::-1]; lam_ = lam_[::-1]

def J_subspace(U):
    Q, _ = np.linalg.qr(U); A = Q @ Q.T
    return np.trace((np.eye(l_) - A) @ Sg)

for t in [0, 0.1, 0.2, 0.4]:
    u = np.cos(t)*V_[:, 4] + np.sin(t)*V_[:, 2]
    print(f"rotación t={t:3.1f}   J = {J_subspace(np.column_stack([V_[:,0], V_[:,1], u])):.4f}")
print("mínimo global sum_{i>m} lambda_i =", lam_[m_:].sum())

**Preguntas para discusión.**
1. ¿Qué ocurre con $\|\boldsymbol{P}_{AE}-\boldsymbol{P}_{PCA}\|_F$ si se repite con otra semilla? ¿Y con $\boldsymbol{W}_1$ individualmente?
2. Si añadimos una activación no lineal en el codificador, ¿sigue valiendo el Teorema 1? (Sugerencia: ver [4] sobre PCA no lineal.)
3. ¿Por qué decimos que "no hay mínimos locales" y aun así el entrenamiento puede ser lento? (Puntos silla.)

---
## I.3 Autoencoders profundos y no lineales

Al usar activaciones no lineales y varias capas, el AE puede aprender **variedades curvas** (*manifolds*) en lugar de subespacios lineales; el Teorema 1 ya no aplica, pero PCA sigue siendo la línea base natural.

**Hipótesis de la variedad.** Los datos de alta dimensión se concentran cerca de una variedad de dimensión mucho menor; el codificador es una *carta* $\mathcal{M}\to\mathbb{R}^m$ y el decodificador su inversa aproximada [6, Cap. 14].

In [ ]:
# ---- Fashion-MNIST ----
(x_tr, y_tr), (x_te, y_te) = keras.datasets.fashion_mnist.load_data()
x_tr = (x_tr.astype("float32")/255.)[..., None]
x_te = (x_te.astype("float32")/255.)[..., None]
x_tr_f = x_tr.reshape(-1, 784); x_te_f = x_te.reshape(-1, 784)
class_names = ["Camiseta","Pantalón","Suéter","Vestido","Abrigo","Sandalia","Camisa","Zapatilla","Bolso","Bota"]
print(x_tr.shape, x_te.shape)

In [ ]:
def build_dense_ae(latent_dim=32, hidden=(256, 128)):
    inp = keras.Input(shape=(784,))
    h = inp
    for u in hidden:
        h = layers.Dense(u, activation="relu")(h)
    z = layers.Dense(latent_dim, name="z")(h)
    h = z
    for u in reversed(hidden):
        h = layers.Dense(u, activation="relu")(h)
    out = layers.Dense(784, activation="sigmoid")(h)
    ae  = keras.Model(inp, out, name="dense_ae")
    enc = keras.Model(inp, z,   name="encoder")
    return ae, enc

ae, enc = build_dense_ae(latent_dim=32)
ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
ae.summary()
h_ae = ae.fit(x_tr_f, x_tr_f, epochs=15, batch_size=256,
              validation_data=(x_te_f, x_te_f), verbose=2)

In [ ]:
def show_recon(model, x, n=10, title=""):
    rec = model.predict(x[:n], verbose=0).reshape(-1, 28, 28)
    fig, ax = plt.subplots(2, n, figsize=(1.2*n, 2.6))
    for i in range(n):
        ax[0,i].imshow(x[i].reshape(28,28), cmap="gray"); ax[0,i].axis("off")
        ax[1,i].imshow(rec[i], cmap="gray");              ax[1,i].axis("off")
    ax[0,0].set_title("original", fontsize=8, loc="left"); ax[1,0].set_title("reconstrucción", fontsize=8, loc="left")
    fig.suptitle(title); plt.show()

show_recon(ae, x_te_f, title="AE profundo (m=32)")

In [ ]:
# ---- Comparación con PCA a igual dimensión latente ----
from sklearn.decomposition import PCA
pca = PCA(n_components=32).fit(x_tr_f)
rec_pca = pca.inverse_transform(pca.transform(x_te_f))
mse_pca = np.mean((x_te_f - rec_pca)**2)
mse_ae  = np.mean((x_te_f - ae.predict(x_te_f, verbose=0))**2)
print(f"MSE PCA(32) = {mse_pca:.5f}   |   MSE AE(32) = {mse_ae:.5f}")

In [ ]:
# ---- Visualización 2-D del espacio latente (AE con m=2) ----
ae2, enc2 = build_dense_ae(latent_dim=2)
ae2.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
ae2.fit(x_tr_f, x_tr_f, epochs=15, batch_size=256, verbose=0)
z2 = enc2.predict(x_te_f, verbose=0)
plt.figure(figsize=(6,5))
sc = plt.scatter(z2[:,0], z2[:,1], c=y_te, cmap="tab10", s=3)
plt.legend(sc.legend_elements()[0], class_names, fontsize=6, markerscale=.7, ncol=2)
plt.title("Espacio latente 2-D de un AE determinista"); plt.show()

---
## I.4 Autoencoders regularizados

Para que un AE sobrecompleto ($m\ge l$) o muy expresivo no aprenda la identidad se añade un **sesgo inductivo**:

| Variante | Mecanismo | Pérdida |
|---|---|---|
| **Denoising (DAE)** [7] | corromper la entrada $\tilde{\boldsymbol{x}}=\boldsymbol{x}+\sigma\boldsymbol{\varepsilon}$ y reconstruir $\boldsymbol{x}$ | $\mathbb{E}\|\boldsymbol{x}-f(g(\tilde{\boldsymbol{x}}))\|^2$ |
| **Sparse** [8] | penalizar actividad del código | $\mathcal{J}+\lambda\|\boldsymbol{z}\|_1$ |
| **Contractivo (CAE)** [9] | penalizar el jacobiano del codificador | $\mathcal{J}+\lambda\|\partial g/\partial\boldsymbol{x}\|_F^2$ |

Todas comparten una idea: **invarianza local** de la representación alrededor de los datos, manteniendo la reconstrucción *sobre* la variedad.

### Teorema 2 (Alain–Bengio, 2014): el autoencoder regularizado estima el *score*

**Definición 2 (Score).** Para una densidad $p(\boldsymbol{x})$ diferenciable, su *score* es $\nabla_{\boldsymbol{x}}\log p(\boldsymbol{x})$, el campo vectorial que apunta hacia regiones de mayor densidad.

> **Teorema 2.** Sea $r_\sigma(\boldsymbol{x})$ la función de reconstrucción **óptima** (con capacidad ilimitada) de un autoencoder denoising entrenado con ruido gaussiano $\tilde{\boldsymbol{x}}=\boldsymbol{x}+\sigma\boldsymbol{\varepsilon}$, $\boldsymbol{\varepsilon}\sim\mathcal{N}(0,I)$, minimizando $\mathbb{E}_{\boldsymbol{x},\boldsymbol{\varepsilon}}\|r(\tilde{\boldsymbol{x}})-\boldsymbol{x}\|^2$. Si $p$ es suave y con soporte adecuado, entonces
> $$
> r_\sigma(\boldsymbol{x})=\boldsymbol{x}+\sigma^2\,\nabla_{\boldsymbol{x}}\log p_\sigma(\boldsymbol{x})
> \quad\Longrightarrow\quad
> \frac{r_\sigma(\boldsymbol{x})-\boldsymbol{x}}{\sigma^2}\xrightarrow[\sigma\to0]{}\nabla_{\boldsymbol{x}}\log p(\boldsymbol{x}),
> $$
> donde $p_\sigma=p*\mathcal{N}(0,\sigma^2I)$ es la densidad suavizada por el ruido.

**Referencias:** Alain & Bengio (2014) [10]; Vincent (2011), *A connection between score matching and denoising autoencoders* [11], que prueba la equivalencia exacta con *score matching* de Hyvärinen (2005) [12].

#### Demostración

El minimizador de un error cuadrático es la esperanza condicional:

$$
r_\sigma(\tilde{\boldsymbol{x}})=\mathbb{E}[\boldsymbol{x}\mid\tilde{\boldsymbol{x}}]=\frac{\int \boldsymbol{x}\,p(\boldsymbol{x})\,\mathcal{N}(\tilde{\boldsymbol{x}};\boldsymbol{x},\sigma^2I)\,d\boldsymbol{x}}{p_\sigma(\tilde{\boldsymbol{x}})}.
$$

Derivamos la densidad suavizada $p_\sigma(\tilde{\boldsymbol{x}})=\int p(\boldsymbol{x})\,\mathcal{N}(\tilde{\boldsymbol{x}};\boldsymbol{x},\sigma^2 I)\,d\boldsymbol{x}$ respecto a $\tilde{\boldsymbol{x}}$ y usamos $\nabla_{\tilde{\boldsymbol{x}}}\mathcal{N}(\tilde{\boldsymbol{x}};\boldsymbol{x},\sigma^2I)=-\tfrac{\tilde{\boldsymbol{x}}-\boldsymbol{x}}{\sigma^2}\mathcal{N}(\tilde{\boldsymbol{x}};\boldsymbol{x},\sigma^2I)$:

$$
\nabla_{\tilde{\boldsymbol{x}}}p_\sigma(\tilde{\boldsymbol{x}})=\frac{1}{\sigma^2}\int(\boldsymbol{x}-\tilde{\boldsymbol{x}})\,p(\boldsymbol{x})\,\mathcal{N}(\tilde{\boldsymbol{x}};\boldsymbol{x},\sigma^2I)\,d\boldsymbol{x}
=\frac{p_\sigma(\tilde{\boldsymbol{x}})}{\sigma^2}\big(\mathbb{E}[\boldsymbol{x}\mid\tilde{\boldsymbol{x}}]-\tilde{\boldsymbol{x}}\big).
$$

Dividiendo por $p_\sigma(\tilde{\boldsymbol{x}})$:

$$
\nabla_{\tilde{\boldsymbol{x}}}\log p_\sigma(\tilde{\boldsymbol{x}})=\frac{r_\sigma(\tilde{\boldsymbol{x}})-\tilde{\boldsymbol{x}}}{\sigma^2}
\;\;\Longleftrightarrow\;\;
r_\sigma(\tilde{\boldsymbol{x}})=\tilde{\boldsymbol{x}}+\sigma^2\nabla\log p_\sigma(\tilde{\boldsymbol{x}}).
$$

Cuando $\sigma\to0$, $p_\sigma\to p$ y se obtiene el límite. $\qquad\blacksquare$

> Esta identidad se conoce como **fórmula de Tweedie** (Robbins 1956; Efron 2011 [13]) y es la base de los **modelos de difusión por score** (Song & Ermon, 2019 [14]): el denoiser *es* el score. Es un puente directo entre esta sesión y la generación moderna.

In [ ]:
# ---- Verificación numérica del Teorema 2 en 1-D (mezcla gaussiana, score exacto conocido) ----
# p(x) = 0.5 N(-2, 0.5^2) + 0.5 N(2, 0.5^2).  Con ruido sigma: p_sigma = mezcla con var 0.5^2 + sigma^2.
mus, s0, sigma = np.array([-2., 2.]), 0.5, 0.4

def sample_p(n):
    k = np.random.randint(0, 2, n)
    return (mus[k] + s0*np.random.randn(n)).astype("float32")

def score_p_sigma(x):
    v = s0**2 + sigma**2
    comp = [np.exp(-0.5*(x-mu)**2/v) for mu in mus]
    num = sum(c*(-(x-mu)/v) for c, mu in zip(comp, mus))
    return num / sum(comp)

xs = sample_p(200_000)
xt = xs + sigma*np.random.randn(len(xs)).astype("float32")

dae = keras.Sequential([keras.Input(shape=(1,)),
                        layers.Dense(128, activation="tanh"),
                        layers.Dense(128, activation="tanh"),
                        layers.Dense(1)])
dae.compile(optimizer=keras.optimizers.Adam(2e-3), loss="mse")
dae.fit(xt[:, None], xs[:, None], epochs=15, batch_size=1024, verbose=0)

grid = np.linspace(-4, 4, 400).astype("float32")
score_est   = (dae.predict(grid[:, None], verbose=0)[:, 0] - grid) / sigma**2
score_exact = score_p_sigma(grid)

plt.figure(figsize=(7,3.4))
plt.plot(grid, score_exact, "k", lw=2, label=r"$\nabla\log p_\sigma(x)$ (exacto)")
plt.plot(grid, score_est, "r--", lw=2, label=r"$(r_\sigma(x)-x)/\sigma^2$ (DAE)")
plt.xlim(-3.5, 3.5); plt.ylim(-15, 15)
plt.title("Teorema 2: el denoiser estima el score"); plt.xlabel("x"); plt.legend(); plt.show()

In [ ]:
# ---- DAE sobre Fashion-MNIST ----
def build_conv_dae(latent_dim=64):
    inp = keras.Input(shape=(28, 28, 1))
    h = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(inp)
    h = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(h)
    h = layers.Flatten()(h)
    z = layers.Dense(latent_dim, name="z")(h)
    h = layers.Dense(7*7*64, activation="relu")(z)
    h = layers.Reshape((7, 7, 64))(h)
    h = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(h)
    h = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(h)
    out = layers.Conv2D(1, 3, padding="same", activation="sigmoid")(h)
    return keras.Model(inp, out, name="conv_dae")

dae_img = build_conv_dae()
dae_img.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")

noise_std = 0.3
def corrupt(x): return np.clip(x + noise_std*np.random.randn(*x.shape).astype("float32"), 0., 1.)

x_tr_noisy = corrupt(x_tr)
x_te_noisy = corrupt(x_te)
dae_img.fit(x_tr_noisy, x_tr, epochs=8, batch_size=256, validation_data=(x_te_noisy, x_te), verbose=2)

n = 8
den = dae_img.predict(x_te_noisy[:n], verbose=0)
fig, ax = plt.subplots(3, n, figsize=(1.3*n, 4))
for i in range(n):
    for r, im in enumerate([x_te[i], x_te_noisy[i], den[i]]):
        ax[r, i].imshow(im.squeeze(), cmap="gray"); ax[r, i].axis("off")
for r, t in enumerate(["limpia", "ruidosa", "DAE"]): ax[r, 0].set_title(t, fontsize=8, loc="left")
plt.show()

In [ ]:
# ---- AE contractivo (CAE): penaliza ||dz/dx||_F^2 calculado con autodiff ----
class ContractiveAE(keras.Model):
    def __init__(self, latent=32, lam=1e-1):
        super().__init__()
        self.lam = lam
        self.enc = keras.Sequential([layers.Dense(128, activation="relu"),
                                     layers.Dense(latent, activation="sigmoid")])
        self.dec = keras.Sequential([layers.Dense(128, activation="relu"),
                                     layers.Dense(784)])                 # logits
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.rec_tracker  = keras.metrics.Mean(name="recon")
        self.jac_tracker  = keras.metrics.Mean(name="jac")

    @property
    def metrics(self): return [self.loss_tracker, self.rec_tracker, self.jac_tracker]

    def train_step(self, data):
        x = data[0] if isinstance(data, (tuple, list)) else data      # x: (B, 784)
        with tf.GradientTape() as outer:
            with tf.GradientTape() as inner:
                inner.watch(x)
                z = self.enc(x)                                       # (B, m)
            J = inner.batch_jacobian(z, x)                            # (B, m, 784)
            jac = tf.reduce_mean(tf.reduce_sum(tf.square(J), axis=[1, 2]))
            logits = self.dec(z)
            rec = tf.reduce_mean(tf.reduce_sum(
                tf.nn.sigmoid_cross_entropy_with_logits(labels=x, logits=logits), axis=1))
            loss = rec + self.lam * jac
        grads = outer.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.loss_tracker.update_state(loss); self.rec_tracker.update_state(rec); self.jac_tracker.update_state(jac)
        return {m.name: m.result() for m in self.metrics}

    def call(self, x): return tf.sigmoid(self.dec(self.enc(x)))

cae = ContractiveAE(latent=32, lam=1e-1)
cae.compile(optimizer=keras.optimizers.Adam(1e-3))
cae.fit(x_tr_f, epochs=5, batch_size=256, verbose=2)

# ---- AE disperso (sparse): regularizador L1 sobre la actividad del código ----
sp_in  = keras.Input(shape=(784,))
sp_z   = layers.Dense(128, activation="relu", activity_regularizer=keras.regularizers.l1(1e-5))(sp_in)
sp_out = layers.Dense(784, activation="sigmoid")(sp_z)
sparse_ae = keras.Model(sp_in, sp_out)
sparse_ae.compile(optimizer="adam", loss="binary_crossentropy")
sparse_ae.fit(x_tr_f, x_tr_f, epochs=8, batch_size=256, verbose=0)

mse = lambda m: np.mean((x_te_f - m.predict(x_te_f, verbose=0))**2)
print(f"MSE test  |  AE denso: {mse(ae):.5f}  |  Contractivo: {mse(cae):.5f}  |  Sparse: {mse(sparse_ae):.5f}")

---
## I.5 Límite del AE como modelo generativo: la motivación de los VAE

Un AE determinista aprende una *codificación*, pero **no una distribución** sobre $\boldsymbol{z}$. Si muestreamos un $\boldsymbol{z}$ arbitrario y lo decodificamos, no hay garantía de que caiga en una región con sentido.

In [ ]:
# Decodificar puntos latentes aleatorios del AE determinista (m=2) -> basura / regiones vacías
zs = np.random.uniform(z2.min(0), z2.max(0), size=(10, 2)).astype("float32")
dec2 = keras.Model(ae2.get_layer("z").output, ae2.output)  # no sirve directamente; reconstruimos decodificador
z_in = keras.Input(shape=(2,)); h = z_in
for lay in ae2.layers[ae2.layers.index(ae2.get_layer("z"))+1:]: h = lay(h)
decoder2 = keras.Model(z_in, h)
imgs = decoder2.predict(zs, verbose=0).reshape(-1, 28, 28)
fig, ax = plt.subplots(1, 10, figsize=(12, 1.4))
for i in range(10): ax[i].imshow(imgs[i], cmap="gray"); ax[i].axis("off")
fig.suptitle("Muestras 'generadas' con un AE determinista: sin estructura probabilística"); plt.show()

**Tres problemas del AE determinista como generador:**

1. **Huecos:** el espacio latente tiene regiones sin datos; el decodificador no está entrenado allí.
2. **Sin densidad:** no hay $p(\boldsymbol{z})$ para muestrear ni para evaluar verosimilitudes.
3. **Sin continuidad garantizada:** puntos latentes cercanos pueden decodificar a imágenes muy distintas.

La solución propuesta por Kingma & Welling [15] y Rezende et al. [16]: **hacer el codificador estocástico y regularizar el espacio latente con un a priori**. Esto es el VAE.

> **Pregunta puente.** ¿Qué *propiedad* debería tener el espacio latente para que muestrear $\boldsymbol{z}\sim\mathcal{N}(0,I)$ y decodificar produjera datos realistas? Intente responder antes de pasar a la Parte II.

---
# PARTE II — Variational Autoencoders (VAE)
---
## II.1 Modelos de variable latente e inferencia variacional

**Definición 3 (Modelo de variable latente).** Un modelo generativo con variable latente $\boldsymbol{z}\in\mathbb{R}^m$ especifica

$$
p_\theta(\boldsymbol{x},\boldsymbol{z})=p_\theta(\boldsymbol{x}\mid\boldsymbol{z})\,p(\boldsymbol{z}),\qquad
\boldsymbol{z}\sim p(\boldsymbol{z})=\mathcal{N}(0,I),\quad \boldsymbol{x}\mid\boldsymbol{z}\sim p_\theta(\boldsymbol{x}\mid\boldsymbol{z}),
$$

donde los parámetros de $p_\theta(\boldsymbol{x}\mid\boldsymbol{z})$ son la salida de una red neuronal (decodificador). La **verosimilitud marginal** (o *evidencia*) es

$$
p_\theta(\boldsymbol{x})=\int p_\theta(\boldsymbol{x}\mid\boldsymbol{z})\,p(\boldsymbol{z})\,d\boldsymbol{z}.
$$

**El problema.** Para decodificadores no lineales esta integral es **intratable**, y también lo es la posterior verdadera $p_\theta(\boldsymbol{z}\mid\boldsymbol{x})=p_\theta(\boldsymbol{x},\boldsymbol{z})/p_\theta(\boldsymbol{x})$. No podemos maximizar $\log p_\theta(\boldsymbol{x})$ directamente ni usar EM clásico.

**Definición 4 (Inferencia variacional amortizada).** Se introduce una familia $q_\phi(\boldsymbol{z}\mid\boldsymbol{x})$ (el **codificador probabilístico**) para aproximar la posterior. *Amortizada* significa que una única red $\phi$ da la aproximación para **todo** $\boldsymbol{x}$, en vez de optimizar parámetros variacionales por cada dato [15, 17].

Habitualmente se toma una gaussiana de covarianza diagonal:

$$
q_\phi(\boldsymbol{z}\mid\boldsymbol{x})=\mathcal{N}\big(\boldsymbol{z};\,\boldsymbol{\mu}_\phi(\boldsymbol{x}),\,\operatorname{diag}(\boldsymbol{\sigma}^2_\phi(\boldsymbol{x}))\big).
$$

| | AE | VAE |
|---|---|---|
| Codificador | $\boldsymbol{z}=g_\phi(\boldsymbol{x})$ (punto) | $q_\phi(\boldsymbol{z}\mid\boldsymbol{x})$ (distribución) |
| Decodificador | $f_\theta(\boldsymbol{z})$ | $p_\theta(\boldsymbol{x}\mid\boldsymbol{z})$ |
| Espacio latente | sin densidad | forzado hacia $p(\boldsymbol{z})$ |
| Objetivo | reconstrucción | cota inferior de $\log p_\theta(\boldsymbol{x})$ |

---
## II.2 La cota inferior de la evidencia (ELBO)

### Teorema 3 (Descomposición de la evidencia y ELBO)

> **Teorema 3.** Sean $p_\theta(\boldsymbol{x},\boldsymbol{z})$ un modelo con posterior $p_\theta(\boldsymbol{z}\mid\boldsymbol{x})$ y $q_\phi(\boldsymbol{z}\mid\boldsymbol{x})$ cualquier densidad con soporte contenido en el de $p_\theta(\boldsymbol{z}\mid\boldsymbol{x})$. Entonces, para todo $\boldsymbol{x}$,
> $$
> \boxed{\;\log p_\theta(\boldsymbol{x})=\underbrace{\mathbb{E}_{q_\phi(\boldsymbol{z}|\boldsymbol{x})}\!\big[\log p_\theta(\boldsymbol{x}\mid\boldsymbol{z})\big]-D_{KL}\!\big(q_\phi(\boldsymbol{z}\mid\boldsymbol{x})\,\|\,p(\boldsymbol{z})\big)}_{\displaystyle\mathcal{L}(\theta,\phi;\boldsymbol{x})\;=\;\text{ELBO}}\;+\;D_{KL}\!\big(q_\phi(\boldsymbol{z}\mid\boldsymbol{x})\,\|\,p_\theta(\boldsymbol{z}\mid\boldsymbol{x})\big)\;}
> $$
> En consecuencia: **(i)** $\log p_\theta(\boldsymbol{x})\ge\mathcal{L}(\theta,\phi;\boldsymbol{x})$ con igualdad **si y solo si** $q_\phi(\boldsymbol{z}\mid\boldsymbol{x})=p_\theta(\boldsymbol{z}\mid\boldsymbol{x})$ casi en todo punto; **(ii)** maximizar el ELBO respecto a $\phi$ equivale a minimizar $D_{KL}(q_\phi\|p_\theta(\cdot|\boldsymbol{x}))$; **(iii)** maximizarlo respecto a $\theta$ eleva una cota inferior de la log-verosimilitud.

**Referencias:** Kingma & Welling (2014) [15, Sec. 2.2]; Jordan et al. (1999) [18]; Blei, Kucukelbur & McAuliffe (2017) [17]; Kingma & Welling (2019) [19, Sec. 2.2].

#### Demostración

**Paso 1 (identidad algebraica).** Por la regla del producto, $p_\theta(\boldsymbol{x})=p_\theta(\boldsymbol{x},\boldsymbol{z})/p_\theta(\boldsymbol{z}\mid\boldsymbol{x})$ para todo $\boldsymbol{z}$ con posterior positiva. Tomando logaritmos y esperanza respecto a $q_\phi(\boldsymbol{z}|\boldsymbol{x})$ (el lado izquierdo no depende de $\boldsymbol{z}$):

$$
\log p_\theta(\boldsymbol{x})=\mathbb{E}_{q}\big[\log p_\theta(\boldsymbol{x},\boldsymbol{z})-\log p_\theta(\boldsymbol{z}\mid\boldsymbol{x})\big].
$$

**Paso 2 (sumar y restar $\log q$).**

$$
\log p_\theta(\boldsymbol{x})=\underbrace{\mathbb{E}_{q}\big[\log p_\theta(\boldsymbol{x},\boldsymbol{z})-\log q_\phi(\boldsymbol{z}|\boldsymbol{x})\big]}_{\mathcal{L}}+\underbrace{\mathbb{E}_{q}\Big[\log\frac{q_\phi(\boldsymbol{z}|\boldsymbol{x})}{p_\theta(\boldsymbol{z}|\boldsymbol{x})}\Big]}_{D_{KL}(q_\phi\|p_\theta(\cdot|\boldsymbol{x}))}.
$$

**Paso 3 (forma reconstrucción–regularización).** Como $p_\theta(\boldsymbol{x},\boldsymbol{z})=p_\theta(\boldsymbol{x}|\boldsymbol{z})p(\boldsymbol{z})$:

$$
\mathcal{L}=\mathbb{E}_q[\log p_\theta(\boldsymbol{x}|\boldsymbol{z})]+\mathbb{E}_q\Big[\log\frac{p(\boldsymbol{z})}{q_\phi(\boldsymbol{z}|\boldsymbol{x})}\Big]=\mathbb{E}_q[\log p_\theta(\boldsymbol{x}|\boldsymbol{z})]-D_{KL}(q_\phi\|p).
$$

**Paso 4 (desigualdad y caso de igualdad).** Por la **desigualdad de Gibbs** (consecuencia de Jensen aplicada a $-\log$ convexa), $D_{KL}(q\|p)\ge0$, con igualdad si y solo si $q=p$ c.t.p. [20, Teor. 2.6.3]. Luego $\log p_\theta(\boldsymbol{x})\ge\mathcal{L}$. $\qquad\blacksquare$

**Demostración alternativa por Jensen** (útil para IWAE, Teorema 6):

$$
\log p_\theta(\boldsymbol{x})=\log\mathbb{E}_{q}\Big[\frac{p_\theta(\boldsymbol{x},\boldsymbol{z})}{q_\phi(\boldsymbol{z}|\boldsymbol{x})}\Big]\ \ge\ \mathbb{E}_{q}\Big[\log\frac{p_\theta(\boldsymbol{x},\boldsymbol{z})}{q_\phi(\boldsymbol{z}|\boldsymbol{x})}\Big]=\mathcal{L}.
$$

La primera demostración es más informativa: **cuantifica exactamente la brecha** ($=D_{KL}$ a la posterior), mientras que Jensen solo da la desigualdad.

**Interpretación.** Dos términos en tensión: *reconstrucción* $\mathbb{E}_q[\log p_\theta(\boldsymbol{x}|\boldsymbol{z})]$ (fidelidad de los datos) y *regularización* $D_{KL}(q\|p)$ (el código no puede alejarse del a priori).

In [ ]:
# ---- Verificación numérica del Teorema 3 en un modelo conjugado con solución cerrada ----
# p(z)=N(0,1), p(x|z)=N(z,1)  =>  p(x)=N(0,2),  p(z|x)=N(x/2, 1/2).  q(z|x)=N(qm, qs^2) arbitrario.
rng = np.random.default_rng(0)
x0, qm, qs = 1.5, 0.2, 1.0

log_px = -0.5*(np.log(2*np.pi*2) + x0**2/2)                       # log p(x) exacta
z = qm + qs*rng.normal(size=2_000_000)
log_joint = -0.5*(np.log(2*np.pi)+z**2) - 0.5*(np.log(2*np.pi)+(x0-z)**2)
log_q     = -0.5*(np.log(2*np.pi*qs**2) + ((z-qm)/qs)**2)
elbo      = np.mean(log_joint - log_q)

pm, ps2 = x0/2, 0.5                                                # posterior exacta
kl_post = np.log(np.sqrt(ps2)/qs) + (qs**2 + (qm-pm)**2)/(2*ps2) - 0.5

print(f"log p(x)            = {log_px:.5f}")
print(f"ELBO                = {elbo:.5f}")
print(f"KL(q || p(z|x))     = {kl_post:.5f}")
print(f"ELBO + KL           = {elbo+kl_post:.5f}   <- coincide con log p(x) (Teorema 3)")
print(f"Brecha log p - ELBO = {log_px-elbo:.5f}   >= 0")

In [ ]:
# ---- La brecha se anula cuando q = posterior verdadera ----
qms = np.linspace(-1, 2, 7)
rows = []
for qm_ in qms:
    qs_ = np.sqrt(0.5)
    z = qm_ + qs_*rng.normal(size=500_000)
    lj = -0.5*(np.log(2*np.pi)+z**2) - 0.5*(np.log(2*np.pi)+(x0-z)**2)
    lq = -0.5*(np.log(2*np.pi*qs_**2)+((z-qm_)/qs_)**2)
    rows.append(np.mean(lj-lq))
plt.figure(figsize=(6,3.2))
plt.plot(qms, rows, "o-", label="ELBO(q)")
plt.axhline(log_px, color="crimson", ls="--", label=r"$\log p(x)$")
plt.axvline(pm, color="gray", ls=":", label="media posterior verdadera")
plt.xlabel(r"media de $q$"); plt.title("ELBO ≤ log p(x), igualdad cuando q = posterior"); plt.legend(); plt.show()

---
## II.3 La pérdida del VAE y la KL gaussiana

Con $q_\phi(\boldsymbol{z}|\boldsymbol{x})=\mathcal{N}(\boldsymbol{\mu},\operatorname{diag}\boldsymbol{\sigma}^2)$ y $p(\boldsymbol{z})=\mathcal{N}(0,I)$ el término de regularización de $\mathcal{L}$ tiene **forma cerrada**; no hace falta muestrear para calcularlo, lo que reduce la varianza del gradiente.

### Teorema 4 (KL entre gaussianas)

> **Teorema 4.** Sean $q=\mathcal{N}(\boldsymbol{\mu},\operatorname{diag}(\boldsymbol{\sigma}^2))$ y $p=\mathcal{N}(0,I)$ en $\mathbb{R}^m$. Entonces
> $$
> D_{KL}(q\,\|\,p)=\frac12\sum_{j=1}^{m}\Big(\mu_j^2+\sigma_j^2-1-\log\sigma_j^2\Big).
> $$
> Más generalmente, para $\mathcal{N}(\boldsymbol{\mu}_1,\Sigma_1)$ y $\mathcal{N}(\boldsymbol{\mu}_2,\Sigma_2)$ en $\mathbb{R}^m$:
> $$
> D_{KL}=\tfrac12\Big[\operatorname{tr}(\Sigma_2^{-1}\Sigma_1)+(\boldsymbol{\mu}_2-\boldsymbol{\mu}_1)^T\Sigma_2^{-1}(\boldsymbol{\mu}_2-\boldsymbol{\mu}_1)-m+\log\tfrac{\det\Sigma_2}{\det\Sigma_1}\Big].
> $$

**Referencias:** Kingma & Welling (2014) [15, Apéndice B]; Cover & Thomas (2006) [20, Cap. 8–9]; Doersch (2016) [21, Sec. 2.2].

#### Demostración

Por factorización, $q$ y $p$ son productos de gaussianas univariadas, y la KL de productos independientes es **aditiva**: $D_{KL}(q\|p)=\sum_j D_{KL}(q_j\|p_j)$. Basta calcular el caso escalar $q_j=\mathcal{N}(\mu,\sigma^2)$, $p_j=\mathcal{N}(0,1)$:

$$
D_{KL}(q_j\|p_j)=\mathbb{E}_{q_j}\big[\log q_j(z)-\log p_j(z)\big].
$$

Usando $\log q_j(z)=-\tfrac12\log(2\pi\sigma^2)-\tfrac{(z-\mu)^2}{2\sigma^2}$ y $\log p_j(z)=-\tfrac12\log(2\pi)-\tfrac{z^2}{2}$:

$$
D_{KL}=-\tfrac12\log\sigma^2-\frac{\mathbb{E}_{q_j}[(z-\mu)^2]}{2\sigma^2}+\frac{\mathbb{E}_{q_j}[z^2]}{2}.
$$

Como $\mathbb{E}_{q_j}[(z-\mu)^2]=\sigma^2$ y $\mathbb{E}_{q_j}[z^2]=\mu^2+\sigma^2$,

$$
D_{KL}=-\tfrac12\log\sigma^2-\tfrac12+\tfrac12(\mu^2+\sigma^2)=\tfrac12\big(\mu^2+\sigma^2-1-\log\sigma^2\big).\qquad\blacksquare
$$

**Observaciones útiles para el diagnóstico:**
- $D_{KL}\ge0$ y vale $0$ si y solo si $\mu=0,\ \sigma=1$ (la dimensión es "ignorada" por el modelo). Esto da un criterio para detectar **dimensiones latentes inactivas** (ver II.8).
- Es **convexa** en $(\mu,\sigma^2)$ y su gradiente es analítico: $\partial D_{KL}/\partial\mu=\mu$, $\ \partial D_{KL}/\partial\sigma^2=\tfrac12(1-1/\sigma^2)$.
- El término $-\log\sigma^2$ impide que $\sigma\to0$: **evita que el VAE degenere en un AE determinista**.

In [ ]:
# ---- Verificación numérica del Teorema 4 en TensorFlow ----
mu = tf.constant([0.7, -1.2]); sig = tf.constant([0.5, 1.8])
kl_closed = 0.5*tf.reduce_sum(mu**2 + sig**2 - 1. - tf.math.log(sig**2))

q = tf.random.normal((2_000_000, 2))*sig + mu
logq = tf.reduce_sum(-0.5*(((q-mu)/sig)**2 + tf.math.log(2*np.pi*sig**2)), axis=1)
logp = tf.reduce_sum(-0.5*(q**2 + np.log(2*np.pi)), axis=1)
kl_mc = tf.reduce_mean(logq - logp)
print(f"KL forma cerrada = {kl_closed.numpy():.5f}   |   KL Monte Carlo = {kl_mc.numpy():.5f}")

# Comprobación del gradiente analítico con autodiff
with tf.GradientTape() as t:
    t.watch([mu, sig])
    kl = 0.5*tf.reduce_sum(mu**2 + sig**2 - 1. - tf.math.log(sig**2))
g_mu, g_sig = t.gradient(kl, [mu, sig])
print("dKL/dmu     autodiff:", g_mu.numpy(),  " analítico:", mu.numpy())
print("dKL/dsigma  autodiff:", g_sig.numpy(), " analítico:", (sig - 1/sig).numpy())

### La función objetivo del VAE

Para un lote de datos, el VAE maximiza el ELBO, o equivalentemente **minimiza**

$$
\boxed{\;\mathcal{J}_{VAE}(\theta,\phi)=-\frac1N\sum_{n=1}^{N}\Big(\underbrace{\mathbb{E}_{q_\phi(\boldsymbol{z}|\boldsymbol{x}^{(n)})}[\log p_\theta(\boldsymbol{x}^{(n)}|\boldsymbol{z})]}_{\text{reconstrucción (MC con 1 muestra)}}-\underbrace{D_{KL}(q_\phi(\boldsymbol{z}|\boldsymbol{x}^{(n)})\|p(\boldsymbol{z}))}_{\text{forma cerrada (Teorema 4)}}\Big)\;}
$$

| Modelo del decodificador | $-\log p_\theta(\boldsymbol{x}|\boldsymbol{z})$ | Uso |
|---|---|---|
| Bernoulli($\hat{\boldsymbol{x}}$) | entropía cruzada binaria (suma sobre píxeles) | imágenes en $[0,1]$ |
| Gaussiana $\mathcal{N}(\hat{\boldsymbol{x}},\sigma^2I)$ | $\tfrac{1}{2\sigma^2}\|\boldsymbol{x}-\hat{\boldsymbol{x}}\|^2+\text{cte}$ | datos continuos |

> **Cuidado frecuente.** La reconstrucción debe **sumarse** sobre las dimensiones de $\boldsymbol{x}$ (no promediarse). Si se promedia, la KL pesa implícitamente $l$ veces más y el modelo tiende al **colapso posterior** (II.8).

**El problema del gradiente.** El gradiente de $\mathbb{E}_{q_\phi}[\cdot]$ respecto a $\phi$ no se puede pasar dentro de la esperanza porque la distribución misma depende de $\phi$. El Teorema 5 resuelve esto.

---
## II.4 El truco de reparametrización

### Teorema 5 (Reparametrización y estimador de gradiente pathwise)

> **Teorema 5.** Sea $\boldsymbol{z}=h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x})$ con $\boldsymbol{\varepsilon}\sim p(\boldsymbol{\varepsilon})$ independiente de $\phi$, donde $h_\phi$ es **diferenciable** respecto a $\phi$. Sea $f$ una función diferenciable que satisface las condiciones de intercambio derivada–integral (convergencia dominada). Entonces
> $$
> \nabla_\phi\,\mathbb{E}_{q_\phi(\boldsymbol{z}|\boldsymbol{x})}\big[f(\boldsymbol{z})\big]=\mathbb{E}_{p(\boldsymbol{\varepsilon})}\Big[\nabla_{\boldsymbol{z}}f(\boldsymbol{z})\big|_{\boldsymbol{z}=h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x})}\;\nabla_\phi h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x})\Big].
> $$
> En particular, para la gaussiana diagonal, $h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x})=\boldsymbol{\mu}_\phi(\boldsymbol{x})+\boldsymbol{\sigma}_\phi(\boldsymbol{x})\odot\boldsymbol{\varepsilon}$, $\boldsymbol{\varepsilon}\sim\mathcal{N}(0,I)$, produce exactamente $\boldsymbol{z}\sim\mathcal{N}(\boldsymbol{\mu},\operatorname{diag}\boldsymbol{\sigma}^2)$.
>
> **Comparación con el estimador *score function* (REINFORCE).** Ambos son insesgados, pero
> $$
> \nabla_\phi\mathbb{E}_{q_\phi}[f]=\mathbb{E}_{q_\phi}\big[f(\boldsymbol{z})\,\nabla_\phi\log q_\phi(\boldsymbol{z}|\boldsymbol{x})\big]
> $$
> típicamente tiene **varianza mucho mayor**, porque no usa la información de pendiente $\nabla_{\boldsymbol{z}}f$ [22, 23].

**Referencias:** Kingma & Welling (2014) [15, Sec. 2.4]; Rezende, Mohamed & Wierstra (2014) [16]; Titsias & Lázaro-Gredilla (2014) [24]; Mohamed et al. (2020), *Monte Carlo gradient estimation in machine learning* [22], que unifica y compara ambos estimadores.

#### Demostración

**Paso 1 (cambio de variable — LOTUS).** Como $\boldsymbol{z}=h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x})$ con $\boldsymbol{\varepsilon}\sim p(\boldsymbol{\varepsilon})$, la ley de $\boldsymbol{z}$ es $q_\phi(\boldsymbol{z}|\boldsymbol{x})$ y, por la *ley del estadístico inconsciente*,

$$
\mathbb{E}_{q_\phi(\boldsymbol{z}|\boldsymbol{x})}[f(\boldsymbol{z})]=\mathbb{E}_{p(\boldsymbol{\varepsilon})}\big[f(h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x}))\big].
$$

**Paso 2 (la medida ya no depende de $\phi$).** El lado derecho integra respecto a $p(\boldsymbol{\varepsilon})$, que **no** depende de $\phi$. Por el *teorema de convergencia dominada* (o regla de Leibniz), podemos intercambiar $\nabla_\phi$ con la esperanza:

$$
\nabla_\phi\,\mathbb{E}_{p(\boldsymbol{\varepsilon})}\big[f(h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x}))\big]=\mathbb{E}_{p(\boldsymbol{\varepsilon})}\big[\nabla_\phi f(h_\phi(\boldsymbol{\varepsilon},\boldsymbol{x}))\big].
$$

**Paso 3 (regla de la cadena).** $\nabla_\phi f(h_\phi)=\nabla_{\boldsymbol{z}}f\cdot\nabla_\phi h_\phi$. $\qquad\blacksquare$

**Paso 4 (el caso gaussiano).** Si $\boldsymbol{\varepsilon}\sim\mathcal{N}(0,I)$, $\boldsymbol{z}=\boldsymbol{\mu}+\boldsymbol{\sigma}\odot\boldsymbol{\varepsilon}$ es combinación afín de una gaussiana, luego es gaussiana con media $\boldsymbol{\mu}$ y covarianza $\operatorname{diag}(\boldsymbol{\sigma}^2)$.

**Ejemplo de cálculo (para $f(z)=z^2$, $z\sim\mathcal{N}(\mu,\sigma^2)$).** $\mathbb{E}[f]=\mu^2+\sigma^2$, así $\partial_\mu=2\mu$, $\partial_\sigma=2\sigma$. Reparametrizado: $\nabla_\mu f=2z$, $\nabla_\sigma f=2z\varepsilon$, y $\mathbb{E}[2z]=2\mu$, $\mathbb{E}[2z\varepsilon]=2\sigma$ ✓.

**Por qué la varianza es menor (intuición).** El estimador *score function* multiplica $f(\boldsymbol{z})$ (que puede ser grande, p.ej. una log-verosimilitud de $-10^3$) por un score ruidoso. El estimador *pathwise* usa $\nabla_{\boldsymbol{z}}f$, que incorpora la estructura diferenciable del decodificador.

**Limitación.** Requiere $\boldsymbol{z}$ **continua** y $h_\phi$ diferenciable; para variables discretas hay que usar relajaciones (Gumbel–Softmax [25]) o estimadores *score function* con *baselines*.

In [ ]:
# ---- Teorema 5: insesgadez y comparación de varianza (reparametrización vs score function) ----
mu0, s0 = tf.constant(0.8), tf.constant(1.3)
M = 2_000_000
eps = tf.random.normal((M,))
z = mu0 + s0*eps

# f(z) = z^2  ->  E[f] = mu^2 + s^2  =>  dmu = 2 mu,  ds = 2 s
g_mu_rep = 2*z                                  # reparametrización (pathwise)
g_mu_sf  = z**2 * (z - mu0)/s0**2               # score function  (REINFORCE)
print(f"dE/dmu exacto = {2*mu0.numpy():.4f}")
print(f"  reparam : media={tf.reduce_mean(g_mu_rep).numpy():.4f}  var={tf.math.reduce_variance(g_mu_rep).numpy():.3f}")
print(f"  scoreFn : media={tf.reduce_mean(g_mu_sf ).numpy():.4f}  var={tf.math.reduce_variance(g_mu_sf ).numpy():.3f}")

# Con autodiff: los gradientes fluyen a (mu, sigma) a través de z
with tf.GradientTape() as tape:
    tape.watch([mu0, s0])
    zz = mu0 + s0*eps
    obj = tf.reduce_mean(zz**2)
print("autodiff (dmu, dsigma):", [g.numpy().round(3) for g in tape.gradient(obj, [mu0, s0])],
      " exactos:", (2*mu0.numpy(), 2*s0.numpy()))

In [ ]:
# ---- Efecto de la varianza al optimizar: ajustar mu de q=N(mu,1) para minimizar E_q[(z-3)^2] ----
def run(estimator, steps=300, lr=0.02, batch=8, seed=0):
    tf.random.set_seed(seed); mu = tf.Variable(-2.0); s = 1.0; tr = []
    for _ in range(steps):
        e = tf.random.normal((batch,)); zz = mu + s*e
        if estimator == "rep":  g = tf.reduce_mean(2*(zz-3.0))
        else:                   g = tf.reduce_mean((zz-3.0)**2 * (zz-mu)/s**2)
        mu.assign_sub(lr*g); tr.append(mu.numpy())
    return np.array(tr)

plt.figure(figsize=(6.5,3.3))
for seed in range(4):
    plt.plot(run("rep", seed=seed), "tab:blue", alpha=.7, label="reparametrización" if seed==0 else None)
    plt.plot(run("sf",  seed=seed), "tab:red",  alpha=.6, label="score function" if seed==0 else None)
plt.axhline(3, color="k", ls=":"); plt.ylim(-4, 8); plt.xlabel("iteración"); plt.ylabel(r"$\mu$")
plt.title("Menor varianza del gradiente ⇒ optimización estable"); plt.legend(); plt.show()

---
## II.5 Implementación de un VAE en TensorFlow

Ensamblamos los Teoremas 3, 4 y 5:

1. **Codificador:** $\boldsymbol{x}\mapsto(\boldsymbol{\mu},\log\boldsymbol{\sigma}^2)$. Se predice $\log\sigma^2$ (no $\sigma$) por **estabilidad numérica** y para no restringir el signo.
2. **Muestreo reparametrizado** (Teorema 5): $\boldsymbol{z}=\boldsymbol{\mu}+\exp(\tfrac12\log\boldsymbol{\sigma}^2)\odot\boldsymbol{\varepsilon}$.
3. **Decodificador:** $\boldsymbol{z}\mapsto$ logits de una Bernoulli.
4. **Pérdida:** $-\text{ELBO}$ con reconstrucción *sumada* y KL cerrada (Teorema 4).

In [ ]:
# ---- MNIST (decodificador Bernoulli con intensidades en [0,1]) ----
(xm_tr, ym_tr), (xm_te, ym_te) = keras.datasets.mnist.load_data()
xm_tr = (xm_tr.astype("float32")/255.)[..., None]
xm_te = (xm_te.astype("float32")/255.)[..., None]
print(xm_tr.shape, xm_te.shape)

In [ ]:
class Sampling(layers.Layer):
    # Truco de reparametrización (Teorema 5): z = mu + sigma * eps
    def call(self, inputs):
        mu, logvar = inputs
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5*logvar) * eps

def build_encoder(latent_dim):
    inp = keras.Input(shape=(28, 28, 1))
    h = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(inp)
    h = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(h)
    h = layers.Flatten()(h)
    h = layers.Dense(128, activation="relu")(h)
    mu     = layers.Dense(latent_dim, name="mu")(h)
    logvar = layers.Dense(latent_dim, name="logvar")(h)
    z = Sampling(name="z")([mu, logvar])
    return keras.Model(inp, [mu, logvar, z], name="encoder")

def build_decoder(latent_dim):
    inp = keras.Input(shape=(latent_dim,))
    h = layers.Dense(128, activation="relu")(inp)
    h = layers.Dense(7*7*64, activation="relu")(h)
    h = layers.Reshape((7, 7, 64))(h)
    h = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(h)
    h = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(h)
    logits = layers.Conv2D(1, 3, padding="same", name="logits")(h)       # logits Bernoulli (más estable)
    return keras.Model(inp, logits, name="decoder")

class VAE(keras.Model):
    def __init__(self, latent_dim, beta=1.0, **kw):
        super().__init__(**kw)
        self.latent_dim = latent_dim
        self.beta = tf.Variable(beta, trainable=False, dtype=tf.float32)
        self.encoder = build_encoder(latent_dim)
        self.decoder = build_decoder(latent_dim)
        self.total_t = keras.metrics.Mean(name="loss")
        self.rec_t   = keras.metrics.Mean(name="recon")
        self.kl_t    = keras.metrics.Mean(name="kl")

    @property
    def metrics(self): return [self.total_t, self.rec_t, self.kl_t]

    def compute_losses(self, x):
        mu, logvar, z = self.encoder(x)
        logits = self.decoder(z)
        # -log p(x|z) para Bernoulli: SUMA sobre píxeles, media sobre el lote
        rec = tf.reduce_mean(tf.reduce_sum(
            tf.nn.sigmoid_cross_entropy_with_logits(labels=x, logits=logits), axis=[1, 2, 3]))
        # KL cerrada (Teorema 4): suma sobre dimensiones latentes
        kl = tf.reduce_mean(0.5*tf.reduce_sum(tf.square(mu) + tf.exp(logvar) - 1. - logvar, axis=1))
        return rec, kl

    def train_step(self, data):
        x = data[0] if isinstance(data, (tuple, list)) else data
        with tf.GradientTape() as tape:
            rec, kl = self.compute_losses(x)
            loss = rec + self.beta*kl                         # beta=1  =>  -ELBO
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.total_t.update_state(loss); self.rec_t.update_state(rec); self.kl_t.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x = data[0] if isinstance(data, (tuple, list)) else data
        rec, kl = self.compute_losses(x)
        self.total_t.update_state(rec + self.beta*kl); self.rec_t.update_state(rec); self.kl_t.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def call(self, x):                                       # reconstrucción determinista (usa mu)
        mu, _, _ = self.encoder(x)
        return tf.sigmoid(self.decoder(mu))

# Empezamos con 2-D para poder visualizar el espacio latente
vae2 = VAE(2)
vae2.compile(optimizer=keras.optimizers.Adam(1e-3))
h2 = vae2.fit(xm_tr, epochs=15, batch_size=128, verbose=2)

In [ ]:
# ---- Curvas de entrenamiento: reconstrucción vs KL ----
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, k, t in zip(ax, ["loss", "recon", "kl"], ["−ELBO", "reconstrucción", "KL"]):
    a.plot(h2.history[k]); a.set_title(t); a.set_xlabel("época")
plt.tight_layout(); plt.show()

In [ ]:
# ---- (1) Espacio latente 2-D coloreado por dígito ----
mu_te, lv_te, _ = vae2.encoder.predict(xm_te, batch_size=1024, verbose=0)
plt.figure(figsize=(6.2, 5.3))
sc = plt.scatter(mu_te[:,0], mu_te[:,1], c=ym_te, cmap="tab10", s=2)
plt.colorbar(sc, label="dígito"); plt.title("Espacio latente del VAE (media posterior)")
plt.xlabel("$z_1$"); plt.ylabel("$z_2$"); plt.show()

# ---- (2) Variedad generada: decodificar una malla del espacio latente ----
from scipy.stats import norm
n = 15
grid_x = norm.ppf(np.linspace(0.05, 0.95, n))
canvas = np.zeros((28*n, 28*n))
for i, yi in enumerate(grid_x[::-1]):
    for j, xi in enumerate(grid_x):
        img = tf.sigmoid(vae2.decoder(np.array([[xi, yi]], dtype="float32"))).numpy().reshape(28, 28)
        canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img
plt.figure(figsize=(7, 7)); plt.imshow(canvas, cmap="gray"); plt.axis("off")
plt.title("Decodificación de una malla latente: espacio continuo y sin huecos"); plt.show()

In [ ]:
# ---- (3) Muestreo generativo z ~ N(0, I) con un VAE de 16 dimensiones ----
vae16 = VAE(16)
vae16.compile(optimizer=keras.optimizers.Adam(1e-3))
h16 = vae16.fit(xm_tr, epochs=15, batch_size=128, verbose=2)

z_prior = tf.random.normal((20, 16))
gen = tf.sigmoid(vae16.decoder(z_prior)).numpy().reshape(-1, 28, 28)
fig, ax = plt.subplots(2, 10, figsize=(12, 2.8))
for i in range(20): ax[i//10, i%10].imshow(gen[i], cmap="gray"); ax[i//10, i%10].axis("off")
fig.suptitle(r"Muestras generadas: $z\sim\mathcal{N}(0,I)$  ->  decodificador"); plt.show()

In [ ]:
# ---- (4) Interpolación latente entre dos dígitos ----
def interpolate(model, xa, xb, steps=10):
    za = model.encoder(xa[None])[0]; zb = model.encoder(xb[None])[0]
    ts = np.linspace(0, 1, steps)[:, None].astype("float32")
    zs = (1-ts)*za.numpy() + ts*zb.numpy()
    return tf.sigmoid(model.decoder(zs)).numpy().reshape(-1, 28, 28)

ia, ib = np.where(ym_te == 3)[0][0], np.where(ym_te == 8)[0][0]
seq = interpolate(vae16, xm_te[ia], xm_te[ib])
fig, ax = plt.subplots(1, 10, figsize=(12, 1.5))
for i in range(10): ax[i].imshow(seq[i], cmap="gray"); ax[i].axis("off")
fig.suptitle("Interpolación latente 3 → 8"); plt.show()

---
## II.6 Cotas más ajustadas: Importance Weighted Autoencoders

El ELBO puede ser una cota **floja** si $q_\phi$ es una mala aproximación de la posterior (Teorema 3: la brecha es exactamente $D_{KL}(q\|p_\theta(\cdot|\boldsymbol{x}))$). Burda et al. proponen usar $K$ muestras con pesos de importancia.

**Definición 5 (Cota IWAE).** Sean $\boldsymbol{z}_1,\dots,\boldsymbol{z}_K\overset{iid}{\sim}q_\phi(\boldsymbol{z}|\boldsymbol{x})$ y $w_k=\dfrac{p_\theta(\boldsymbol{x},\boldsymbol{z}_k)}{q_\phi(\boldsymbol{z}_k|\boldsymbol{x})}$. Se define

$$
\mathcal{L}_K(\boldsymbol{x})=\mathbb{E}_{\boldsymbol{z}_{1:K}}\Big[\log\frac1K\sum_{k=1}^{K}w_k\Big].
$$

Nótese que $\mathcal{L}_1=\text{ELBO}$.

### Teorema 6 (Monotonía de la cota IWAE; Burda et al., 2016)

> **Teorema 6.** Si los pesos de importancia $w_k$ están acotados (o más generalmente $\mathbb{E}|\log w|<\infty$), entonces
> $$
> \mathcal{L}_1\ \le\ \mathcal{L}_2\ \le\ \cdots\ \le\ \mathcal{L}_K\ \le\ \log p_\theta(\boldsymbol{x}),
> $$
> y además $\mathcal{L}_K\to\log p_\theta(\boldsymbol{x})$ cuando $K\to\infty$.

**Referencias:** Burda, Grosse & Salakhutdinov (2016) [26, Teor. 1 y 2]; Cremer, Morris & Duvenaud (2017), *Reinterpreting importance-weighted autoencoders* [27]; Domke & Sheldon (2018) [28], que cuantifican la tasa de ajuste de la cota.

#### Demostración

**(a) $\mathcal{L}_K\le\log p_\theta(\boldsymbol{x})$.** El promedio $\hat{p}_K=\frac1K\sum_k w_k$ es un estimador **insesgado** de $p_\theta(\boldsymbol{x})$, pues $\mathbb{E}_q[w_k]=\int p_\theta(\boldsymbol{x},\boldsymbol{z})\,d\boldsymbol{z}=p_\theta(\boldsymbol{x})$. Por Jensen ($\log$ cóncava): $\mathbb{E}[\log\hat{p}_K]\le\log\mathbb{E}[\hat{p}_K]=\log p_\theta(\boldsymbol{x})$.

**(b) Monotonía $\mathcal{L}_{K+1}\ge\mathcal{L}_K$.** Sea $I\subset\{1,\dots,K+1\}$ un subconjunto uniforme de tamaño $K$. Entonces el promedio completo es el promedio de los $K{+}1$ promedios parciales:

$$
\frac{1}{K+1}\sum_{k=1}^{K+1}w_k=\mathbb{E}_I\Big[\frac1K\sum_{i\in I}w_i\Big].
$$

Aplicando $\log$ y la desigualdad de Jensen sobre $\mathbb{E}_I$:

$$
\log\frac{1}{K+1}\sum_{k=1}^{K+1}w_k\ \ge\ \mathbb{E}_I\Big[\log\frac1K\sum_{i\in I}w_i\Big].
$$

Tomando esperanza sobre $\boldsymbol{z}_{1:K+1}$ y notando que, para cada $I$ fijo, los $K$ elementos son iid con la misma ley, el lado derecho es exactamente $\mathcal{L}_K$. Luego $\mathcal{L}_{K+1}\ge\mathcal{L}_K$.

**(c) Convergencia.** Por la ley fuerte de los grandes números, $\hat{p}_K\to p_\theta(\boldsymbol{x})$ casi seguramente; con $w_k$ acotados, la convergencia dominada da $\mathbb{E}[\log\hat{p}_K]\to\log p_\theta(\boldsymbol{x})$. $\qquad\blacksquare$

**Implicación práctica.** Con $K$ grande, **no es necesario que $q_\phi$ sea una buena posterior**: la cota se ajusta sola. Costo: $K$ veces más cómputo del decodificador y gradientes de **menor** relación señal/ruido para el codificador (Rainforth et al., 2018 [29]).

In [ ]:
# ---- Verificación del Teorema 6: L_K crece monótonamente hacia log p(x) (modelo conjugado) ----
rng = np.random.default_rng(0)
x0, qm, qs = 1.5, 0.2, 1.0
log_px = -0.5*(np.log(2*np.pi*2) + x0**2/2)

def L_K(K, R=60000):
    z = qm + qs*rng.normal(size=(R, K))
    logw = (-0.5*(np.log(2*np.pi)+z**2)) + (-0.5*(np.log(2*np.pi)+(x0-z)**2)) \
           - (-0.5*(np.log(2*np.pi*qs**2) + ((z-qm)/qs)**2))
    mx = logw.max(1, keepdims=True)
    return np.mean(mx[:, 0] + np.log(np.mean(np.exp(logw-mx), axis=1)))

Ks = [1, 2, 5, 10, 20, 50, 100, 200, 500]
vals = [L_K(K) for K in Ks]
plt.figure(figsize=(6.3, 3.4))
plt.semilogx(Ks, vals, "o-", label=r"$\mathcal{L}_K$")
plt.axhline(log_px, color="crimson", ls="--", label=r"$\log p(x)$")
plt.xlabel("K (nº de muestras)"); plt.title("Teorema 6: cota IWAE creciente en K"); plt.legend(); plt.show()
print({K: round(v, 4) for K, v in zip(Ks, vals)}, "| log p(x) =", round(log_px, 4))

In [ ]:
# ---- Estimar L_K en el VAE entrenado (MNIST, decodificador Bernoulli) ----
def iwae_bound(model, x, K=50):
    B = tf.shape(x)[0]
    mu, logvar, _ = model.encoder(x)
    mu, logvar = mu[:, None, :], logvar[:, None, :]                       # (B,1,m)
    eps = tf.random.normal((B, K, model.latent_dim))
    z = mu + tf.exp(0.5*logvar)*eps                                       # (B,K,m)
    logits = model.decoder(tf.reshape(z, (-1, model.latent_dim)))
    logits = tf.reshape(logits, (B, K, 28, 28, 1))
    xr = tf.repeat(x[:, None], K, axis=1)
    log_px_z = -tf.reduce_sum(tf.nn.sigmoid_cross_entropy_with_logits(labels=xr, logits=logits), axis=[2, 3, 4])
    log_pz   = tf.reduce_sum(-0.5*(z**2 + np.log(2*np.pi)), axis=2)
    log_qz   = tf.reduce_sum(-0.5*(eps**2 + np.log(2*np.pi) + logvar), axis=2)
    logw = log_px_z + log_pz - log_qz                                     # (B,K)
    return tf.reduce_mean(tf.reduce_logsumexp(logw, axis=1) - np.log(K))

xb = tf.constant(xm_te[:500])
for K in [1, 5, 50, 500]:
    print(f"K={K:4d}   L_K = {iwae_bound(vae16, xb, K).numpy():8.3f} nats")

---
## II.7 Tasa–distorsión y $\beta$-VAE

El ELBO mezcla dos cantidades con significado informacional. Definimos:

- **Distorsión:** $D=-\mathbb{E}_{p_{\text{data}}(\boldsymbol{x})}\mathbb{E}_{q_\phi(\boldsymbol{z}|\boldsymbol{x})}[\log p_\theta(\boldsymbol{x}|\boldsymbol{z})]$ (error de reconstrucción esperado).
- **Tasa:** $R=\mathbb{E}_{p_{\text{data}}(\boldsymbol{x})}\big[D_{KL}(q_\phi(\boldsymbol{z}|\boldsymbol{x})\|p(\boldsymbol{z}))\big]$ (coste de describir $\boldsymbol{z}$ respecto al a priori).

### Teorema 7 (ELBO como compromiso tasa–distorsión; Alemi et al., 2018)

> **Teorema 7.** Sea $I_q(\boldsymbol{x};\boldsymbol{z})$ la información mutua bajo $p_{\text{data}}(\boldsymbol{x})\,q_\phi(\boldsymbol{z}|\boldsymbol{x})$ y $H$ la entropía de los datos. Entonces:
> $$
> \text{(i)}\quad R=I_q(\boldsymbol{x};\boldsymbol{z})+D_{KL}\big(q_\phi(\boldsymbol{z})\,\|\,p(\boldsymbol{z})\big)\ \ge\ I_q(\boldsymbol{x};\boldsymbol{z}),\qquad q_\phi(\boldsymbol{z}):=\mathbb{E}_{p_{\text{data}}}[q_\phi(\boldsymbol{z}|\boldsymbol{x})],
> $$
> $$
> \text{(ii)}\quad H-D\ \le\ I_q(\boldsymbol{x};\boldsymbol{z})\ \le\ R,\qquad
> \text{(iii)}\quad -\text{ELBO}_{\text{promedio}}=D+R .
> $$
> Por tanto, minimizar $D+\beta R$ recorre la **frontera tasa–distorsión**; el VAE estándar es el punto $\beta=1$, y el **$\beta$-VAE** (Higgins et al., 2017) con $\beta>1$ comprime más el código a costa de la reconstrucción.

**Referencias:** Alemi et al. (2018) [30, Secc. 2–3]; Higgins et al. (2017) [31]; Hoffman & Johnson (2016), *ELBO surgery* [32]; Burgess et al. (2018) [33].

#### Demostración

**(i) Descomposición de la tasa.** Con $q_\phi(\boldsymbol{z})=\int p_{\text{data}}(\boldsymbol{x})q_\phi(\boldsymbol{z}|\boldsymbol{x})d\boldsymbol{x}$ (posterior agregada),

$$
R=\mathbb{E}_{p_{\text{data}}}\mathbb{E}_{q}\Big[\log\frac{q_\phi(\boldsymbol{z}|\boldsymbol{x})}{p(\boldsymbol{z})}\Big]
=\mathbb{E}\Big[\log\frac{q_\phi(\boldsymbol{z}|\boldsymbol{x})}{q_\phi(\boldsymbol{z})}\Big]+\mathbb{E}\Big[\log\frac{q_\phi(\boldsymbol{z})}{p(\boldsymbol{z})}\Big]
=I_q(\boldsymbol{x};\boldsymbol{z})+D_{KL}(q_\phi(\boldsymbol{z})\|p(\boldsymbol{z})).
$$

El segundo término es $\ge0$ (Gibbs), de donde $R\ge I_q$.

**(ii) Cota inferior de la información mutua.** $I_q(\boldsymbol{x};\boldsymbol{z})=H(\boldsymbol{x})-H_q(\boldsymbol{x}|\boldsymbol{z})$. Como la entropía condicional verdadera se acota por la entropía cruzada del decodificador, $H_q(\boldsymbol{x}|\boldsymbol{z})\le D$ (por Gibbs, ya que $\mathbb{E}_{q(\boldsymbol{x}|\boldsymbol{z})}\log\frac{q(\boldsymbol{x}|\boldsymbol{z})}{p_\theta(\boldsymbol{x}|\boldsymbol{z})}\ge0$). Luego $I_q\ge H-D$.

**(iii) Identidad del ELBO.** Promediando el Teorema 3 sobre los datos: $-\text{ELBO}=D+R$ por definición de ambos términos. $\qquad\blacksquare$

**Consecuencias importantes:**
- **Fallo del ELBO como criterio único:** existen modelos con el **mismo ELBO** pero tasas y distorsiones muy distintas: $(D,R)$ y $(D+\delta,R-\delta)$ tienen igual $D+R$. Un VAE con $R\approx0$ (código inútil) puede tener un ELBO similar a otro con código informativo.
- **$\beta$-VAE** minimiza $D+\beta R$ = Lagrangiano del problema $\min D$ s.a. $R\le C$; barrer $\beta$ recorre la curva $D(R)$.
- La cota $R\ge I_q(\boldsymbol{x};\boldsymbol{z})$ explica por qué un $\beta$ grande **limita la información** que el código puede portar sobre $\boldsymbol{x}$.

In [ ]:
# ---- Barrido de beta: curva tasa–distorsión empírica (Teorema 7) ----
betas = [0.1, 0.5, 1.0, 2.0, 4.0, 8.0]
results, models_beta = [], {}
for b in betas:
    tf.random.set_seed(0)
    m_ = VAE(8, beta=b); m_.compile(optimizer=keras.optimizers.Adam(1e-3))
    m_.fit(xm_tr, epochs=6, batch_size=128, verbose=0)
    D_, R_ = [], []
    for i in range(0, len(xm_te), 1000):
        d, r = m_.compute_losses(tf.constant(xm_te[i:i+1000])); D_.append(d.numpy()); R_.append(r.numpy())
    results.append((b, np.mean(D_), np.mean(R_))); models_beta[b] = m_      # (beta, D, R)
    print(f"beta={b:4.1f}   D (recon)={np.mean(D_):7.2f}   R (KL)={np.mean(R_):6.2f}   D+R={np.mean(D_)+np.mean(R_):7.2f}")

rd = np.array(results)
plt.figure(figsize=(6.3, 3.8))
plt.plot(rd[:, 2], rd[:, 1], "o-")
for b, d, r in rd: plt.annotate(rf"$\beta$={b:g}", (r, d), textcoords="offset points", xytext=(5, 5), fontsize=8)
plt.xlabel("Tasa R = KL (nats)"); plt.ylabel("Distorsión D (nats)")
plt.title("Frontera tasa–distorsión del VAE (Teorema 7)"); plt.show()

In [ ]:
# ---- Efecto cualitativo de beta (reutiliza los modelos entrenados arriba) ----
fig, ax = plt.subplots(2, 6, figsize=(12, 4.2))
z_fix = tf.random.normal((1, 8), seed=1)
for j, b in enumerate(betas):
    mm = models_beta[b]
    ax[0, j].imshow(mm(xm_te[:1]).numpy().reshape(28, 28), cmap="gray"); ax[0, j].set_title(rf"$\beta$={b:g}"); ax[0, j].axis("off")
    ax[1, j].imshow(tf.sigmoid(mm.decoder(z_fix)).numpy().reshape(28, 28), cmap="gray"); ax[1, j].axis("off")
fig.suptitle("Fila 1: reconstrucción | Fila 2: muestra del prior (mismo z)"); plt.show()

---
## II.8 Diagnóstico: colapso posterior, dimensiones activas y VAE condicional

### Colapso posterior (*posterior collapse*)

Ocurre cuando $q_\phi(\boldsymbol{z}|\boldsymbol{x})\approx p(\boldsymbol{z})$ para todo $\boldsymbol{x}$: el decodificador **ignora** $\boldsymbol{z}$ y modela $p(\boldsymbol{x})$ por sí solo (típico con decodificadores autorregresivos potentes). De la forma de $R$ (Teorema 7): $R\to0\Rightarrow I_q(\boldsymbol{x};\boldsymbol{z})\to0$.

**Diagnóstico por dimensión** (consecuencia del Teorema 4): una dimensión $j$ está *activa* si $\operatorname{Var}_{\boldsymbol{x}}\big[\mu_j(\boldsymbol{x})\big]>\delta$ (Burda et al. [26] usan $\delta=0.01$).

**Remedios:** *KL annealing* (subir $\beta$ de 0 a 1 gradualmente, Bowman et al. 2016 [34]); *free bits* (Kingma et al. 2016 [35]); debilitar el decodificador; $\beta<1$. Ver también Lucas et al. (2019) [36] y Dai & Wipf (2019) [37] para el análisis teórico del colapso y la mejora en dos etapas.

In [ ]:
# ---- Dimensiones latentes activas: Var_x[mu_j(x)] > 0.01 ----
def active_units(model, x, thr=0.01):
    mu, _, _ = model.encoder.predict(x, batch_size=1024, verbose=0)
    v = mu.var(axis=0); return v, int((v > thr).sum())

big = VAE(32); big.compile(optimizer=keras.optimizers.Adam(1e-3))
big.fit(xm_tr, epochs=10, batch_size=128, verbose=0)
v, n_act = active_units(big, xm_te)
plt.figure(figsize=(7, 2.8))
plt.bar(range(32), v); plt.axhline(0.01, color="crimson", ls="--", label="umbral 0.01")
plt.xlabel("dimensión latente"); plt.ylabel(r"Var$_x[\mu_j]$"); plt.legend()
plt.title(f"Unidades activas: {n_act}/32  (el VAE usa solo las dimensiones que necesita)"); plt.show()

In [ ]:
# ---- KL annealing: callback que sube beta de 0 a 1 ----
class KLAnneal(keras.callbacks.Callback):
    def __init__(self, warmup_epochs=5): super().__init__(); self.w = warmup_epochs
    def on_epoch_begin(self, epoch, logs=None):
        self.model.beta.assign(min(1.0, epoch/self.w))

va = VAE(16); va.compile(optimizer=keras.optimizers.Adam(1e-3))
ha = va.fit(xm_tr, epochs=10, batch_size=128, callbacks=[KLAnneal(5)], verbose=2)

### VAE condicional (CVAE)

Condicionamos en una etiqueta $\boldsymbol{y}$ (p.ej. el dígito). El ELBO se mantiene, con todas las densidades condicionadas:

$$
\log p_\theta(\boldsymbol{x}|\boldsymbol{y})\ \ge\ \mathbb{E}_{q_\phi(\boldsymbol{z}|\boldsymbol{x},\boldsymbol{y})}\big[\log p_\theta(\boldsymbol{x}|\boldsymbol{z},\boldsymbol{y})\big]-D_{KL}\big(q_\phi(\boldsymbol{z}|\boldsymbol{x},\boldsymbol{y})\|p(\boldsymbol{z}|\boldsymbol{y})\big).
$$

(La demostración es idéntica al Teorema 3 con $\boldsymbol{y}$ como variable de condicionamiento fija; Sohn et al., 2015 [38].) Permite **generación controlada**: elegir la clase y variar el estilo con $\boldsymbol{z}$.

In [ ]:
# ---- CVAE: generar un dígito elegido ----
class CVAE(keras.Model):
    def __init__(self, latent_dim=8, n_cls=10):
        super().__init__()
        self.latent_dim, self.n_cls = latent_dim, n_cls
        xin = keras.Input((28, 28, 1)); yin = keras.Input((n_cls,))
        h = layers.Flatten()(xin); h = layers.Concatenate()([h, yin])
        h = layers.Dense(256, activation="relu")(h)
        mu = layers.Dense(latent_dim)(h); lv = layers.Dense(latent_dim)(h)
        self.enc = keras.Model([xin, yin], [mu, lv])
        zin = keras.Input((latent_dim,)); yin2 = keras.Input((n_cls,))
        h = layers.Concatenate()([zin, yin2]); h = layers.Dense(256, activation="relu")(h)
        out = layers.Dense(784)(h)
        self.dec = keras.Model([zin, yin2], out)
        self.tr = keras.metrics.Mean(name="loss")
    @property
    def metrics(self): return [self.tr]
    def train_step(self, data):
        (x, y), _ = data
        with tf.GradientTape() as tape:
            mu, lv = self.enc([x, y]); z = mu + tf.exp(0.5*lv)*tf.random.normal(tf.shape(mu))
            logits = tf.reshape(self.dec([z, y]), (-1, 28, 28, 1))
            rec = tf.reduce_mean(tf.reduce_sum(tf.nn.sigmoid_cross_entropy_with_logits(labels=x, logits=logits), [1, 2, 3]))
            kl = tf.reduce_mean(0.5*tf.reduce_sum(mu**2 + tf.exp(lv) - 1. - lv, 1))
            loss = rec + kl
        self.optimizer.apply_gradients(zip(tape.gradient(loss, self.trainable_variables), self.trainable_variables))
        self.tr.update_state(loss); return {"loss": self.tr.result()}

y_oh = keras.utils.to_categorical(ym_tr, 10).astype("float32")
cvae = CVAE(); cvae.compile(optimizer=keras.optimizers.Adam(1e-3))
cvae.fit((xm_tr, y_oh), y_oh, epochs=12, batch_size=128, verbose=2)

fig, ax = plt.subplots(10, 8, figsize=(8, 10))
for d in range(10):
    z = tf.random.normal((8, cvae.latent_dim)); yy = tf.one_hot([d]*8, 10)
    im = tf.sigmoid(cvae.dec([z, yy])).numpy().reshape(8, 28, 28)
    for j in range(8): ax[d, j].imshow(im[j], cmap="gray"); ax[d, j].axis("off")
fig.suptitle("CVAE: cada fila = una clase; columnas = estilos (z)"); plt.show()

### Aplicación: detección de anomalías con VAE

La log-verosimilitud aproximada (ELBO o IWAE) sirve como **puntuación de anomalía**: entradas atípicas tendrán un ELBO bajo. *Advertencia:* los modelos profundos pueden asignar alta verosimilitud a datos fuera de distribución (Nalisnick et al., 2019 [39]); conviene contrastar con otras métricas.

In [ ]:
# ---- Anomalías: entrenado con MNIST, se puntúa con ELBO a MNIST (in) vs Fashion-MNIST (out) ----
def elbo_score(model, x):
    out = []
    for i in range(0, len(x), 1000):
        xb = tf.constant(x[i:i+1000])
        mu, lv, z = model.encoder(xb)
        logits = model.decoder(z)
        rec = tf.reduce_sum(tf.nn.sigmoid_cross_entropy_with_logits(labels=xb, logits=logits), [1, 2, 3])
        kl = 0.5*tf.reduce_sum(mu**2 + tf.exp(lv) - 1. - lv, 1)
        out.append((-(rec + kl)).numpy())            # ELBO por muestra
    return np.concatenate(out)

s_in  = elbo_score(vae16, xm_te)
s_out = elbo_score(vae16, x_te[:len(xm_te)])         # Fashion-MNIST como anomalía
from sklearn.metrics import roc_auc_score
y_true = np.r_[np.zeros_like(s_in), np.ones_like(s_out)]
print("AUROC (ELBO como detector):", roc_auc_score(y_true, -np.r_[s_in, s_out]))
plt.figure(figsize=(6, 3)); plt.hist(s_in, 60, alpha=.6, label="MNIST (in)"); plt.hist(s_out, 60, alpha=.6, label="Fashion (out)")
plt.xlabel("ELBO"); plt.legend(); plt.title("Detección de anomalías"); plt.show()

---
## Ejercicios propuestos

**Teóricos**

1. **(Teorema 1)** Complete el Paso 3 mostrando que $\boldsymbol{A}=\boldsymbol{W}_2\boldsymbol{W}_1$ es idempotente. ¿Qué falla si $\Sigma$ tiene autovalores repetidos? (Pista: el subespacio invariante ya no es único.)
2. **(Teorema 2)** Demuestre la fórmula de Tweedie para ruido no isotrópico $\mathcal{N}(0,\Sigma_\eta)$: $r(\tilde{\boldsymbol{x}})=\tilde{\boldsymbol{x}}+\Sigma_\eta\nabla\log p_\eta(\tilde{\boldsymbol{x}})$.
3. **(Teorema 3)** Pruebe que si $q_\phi(\boldsymbol{z}|\boldsymbol{x})$ pertenece a una familia que *contiene* a la posterior verdadera, el máximo del ELBO sobre $\phi$ es $\log p_\theta(\boldsymbol{x})$.
4. **(Teorema 4)** Derive la KL entre $\mathcal{N}(\boldsymbol{\mu}_1,\Sigma_1)$ y $\mathcal{N}(\boldsymbol{\mu}_2,\Sigma_2)$ con covarianza completa.
5. **(Teorema 5)** Encuentre la reparametrización para una distribución **Laplace** y una **log-normal**. ¿Y para una **Gamma**? (Ver [24].)
6. **(Teorema 6)** Pruebe que $\mathcal{L}_K$ es el ELBO de un modelo extendido con $K$ muestras y un $q$ aumentado (Cremer et al. [27]).
7. **(Teorema 7)** Muestre un ejemplo de dos modelos con igual ELBO y tasas distintas.

**Computacionales (TensorFlow)**

8. Reemplace el decodificador Bernoulli por uno **gaussiano con varianza aprendida** $\mathcal{N}(\mu_\theta(\boldsymbol{z}),\sigma_\theta^2(\boldsymbol{z}))$ y compare en Fashion-MNIST. ¿Qué ocurre con $\sigma_\theta$?
9. Implemente un **IWAE** completo (entrene con $\mathcal{L}_{K=5}$) y compare $\mathcal{L}_{5000}$ en test con el VAE estándar.
10. Implemente el **$\beta$-VAE con capacidad controlada** (Burgess et al. [33]): pérdida $D+\gamma|R-C|$ con $C$ creciente.
11. Implemente **free bits** y compare las unidades activas con KL annealing.
12. Entrene un VAE sobre **datos tabulares financieros** (rendimientos de activos) y úselo para detectar días atípicos.

---
## Resumen de la sesión

| # | Resultado | Idea clave | Uso práctico |
|---|---|---|---|
| **T1** | AE lineal = PCA (Baldi–Hornik) | sin mínimos locales, solo sillas | línea base, inicialización |
| **T2** | DAE estima el score (Alain–Bengio, Vincent) | $r-x\approx\sigma^2\nabla\log p$ | modelos de difusión |
| **T3** | $\log p=\text{ELBO}+D_{KL}(q\|p_\theta)$ | cota exacta y brecha cuantificada | función objetivo del VAE |
| **T4** | KL gaussiana cerrada | sin muestreo, gradiente exacto | término regularizador |
| **T5** | Reparametrización | gradiente pathwise de baja varianza | entrenamiento end-to-end |
| **T6** | IWAE: $\mathcal{L}_1\le\mathcal{L}_K\le\log p$ | más muestras, cota más ajustada | evaluación de verosimilitud |
| **T7** | $-\text{ELBO}=D+R$, $R\ge I(x;z)$ | compromiso tasa–distorsión | $\beta$-VAE, diagnóstico |

---
# Referencias

> Las referencias de **autores, título, revista/conferencia, año y identificador arXiv/DOI** de los trabajos centrales (T1, T2, T3, T5, T6, T7) fueron verificadas en línea (arXiv, JMLR, ScienceDirect, ICLR/ICML proceedings). Las demás corresponden a trabajos clásicos ampliamente citados; se recomienda confirmar páginas y números exactos antes de publicar.

## Autoencoders lineales y PCA (Teorema 1)
- **[1]** Baldi, P. & Hornik, K. (1989). *Neural networks and principal component analysis: Learning from examples without local minima*. **Neural Networks**, 2(1), 53–58. DOI: 10.1016/0893-6080(89)90014-2
- **[2]** Bourlard, H. & Kamp, Y. (1988). *Auto-association by multilayer perceptrons and singular value decomposition*. **Biological Cybernetics**, 59, 291–294.
- **[3]** Kunin, D., Bloom, J. M., Goeva, A. & Seed, C. (2019). *Loss landscapes of regularized linear autoencoders*. **ICML 2019**. arXiv:1901.08168.
- **[4]** Kramer, M. A. (1991). *Nonlinear principal component analysis using autoassociative neural networks*. **AIChE Journal**, 37(2), 233–243.
- **[5]** Eckart, C. & Young, G. (1936). *The approximation of one matrix by another of lower rank*. **Psychometrika**, 1, 211–218. / Jolliffe, I. T. (2002). *Principal Component Analysis*, 2nd ed. Springer.
- **[6]** Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*. MIT Press. Cap. 14 (Autoencoders), Cap. 20 (Deep Generative Models). https://www.deeplearningbook.org

## Autoencoders regularizados y score (Teorema 2)
- **[7]** Vincent, P., Larochelle, H., Lajoie, I., Bengio, Y. & Manzagol, P.-A. (2010). *Stacked denoising autoencoders*. **JMLR**, 11, 3371–3408.
- **[8]** Ng, A. (2011). *Sparse autoencoder*. CS294A Lecture Notes, Stanford.
- **[9]** Rifai, S., Vincent, P., Muller, X., Glorot, X. & Bengio, Y. (2011). *Contractive auto-encoders: Explicit invariance during feature extraction*. **ICML 2011**.
- **[10]** Alain, G. & Bengio, Y. (2014). *What regularized auto-encoders learn from the data-generating distribution*. **JMLR**, 15, 3743–3773. arXiv:1211.4246.
- **[11]** Vincent, P. (2011). *A connection between score matching and denoising autoencoders*. **Neural Computation**, 23(7), 1661–1674.
- **[12]** Hyvärinen, A. (2005). *Estimation of non-normalized statistical models by score matching*. **JMLR**, 6, 695–709.
- **[13]** Efron, B. (2011). *Tweedie's formula and selection bias*. **JASA**, 106(496), 1602–1614.
- **[14]** Song, Y. & Ermon, S. (2019). *Generative modeling by estimating gradients of the data distribution*. **NeurIPS 2019**. arXiv:1907.05600.

## VAE: fundamentos y ELBO (Teoremas 3, 4, 5)
- **[15]** Kingma, D. P. & Welling, M. (2014). *Auto-Encoding Variational Bayes*. **ICLR 2014**. arXiv:1312.6114.
- **[16]** Rezende, D. J., Mohamed, S. & Wierstra, D. (2014). *Stochastic backpropagation and approximate inference in deep generative models*. **ICML 2014**, PMLR 32. arXiv:1401.4082.
- **[17]** Blei, D. M., Kucukelbir, A. & McAuliffe, J. D. (2017). *Variational inference: A review for statisticians*. **JASA**, 112(518), 859–877. arXiv:1601.00670.
- **[18]** Jordan, M. I., Ghahramani, Z., Jaakkola, T. S. & Saul, L. K. (1999). *An introduction to variational methods for graphical models*. **Machine Learning**, 37, 183–233.
- **[19]** Kingma, D. P. & Welling, M. (2019). *An introduction to variational autoencoders*. **Foundations and Trends in Machine Learning**, 12(4), 307–392. arXiv:1906.02691. *(Lectura central recomendada)*
- **[20]** Cover, T. M. & Thomas, J. A. (2006). *Elements of Information Theory*, 2nd ed. Wiley.
- **[21]** Doersch, C. (2016). *Tutorial on variational autoencoders*. arXiv:1606.05908.
- **[22]** Mohamed, S., Rosca, M., Figurnov, M. & Mnih, A. (2020). *Monte Carlo gradient estimation in machine learning*. **JMLR**, 21(132), 1–62. arXiv:1906.10652.
- **[23]** Williams, R. J. (1992). *Simple statistical gradient-following algorithms for connectionist reinforcement learning*. **Machine Learning**, 8, 229–256.
- **[24]** Titsias, M. & Lázaro-Gredilla, M. (2014). *Doubly stochastic variational Bayes for non-conjugate inference*. **ICML 2014**.
- **[25]** Jang, E., Gu, S. & Poole, B. (2017). *Categorical reparameterization with Gumbel-Softmax*. **ICLR 2017**. arXiv:1611.01144. / Maddison, C., Mnih, A. & Teh, Y. W. (2017). *The Concrete distribution*. **ICLR 2017**.

## Cotas ajustadas y tasa–distorsión (Teoremas 6, 7)
- **[26]** Burda, Y., Grosse, R. & Salakhutdinov, R. (2016). *Importance weighted autoencoders*. **ICLR 2016**. arXiv:1509.00519.
- **[27]** Cremer, C., Morris, Q. & Duvenaud, D. (2017). *Reinterpreting importance-weighted autoencoders*. **ICLR 2017 Workshop**. arXiv:1704.02916.
- **[28]** Domke, J. & Sheldon, D. (2018). *Importance weighting and variational inference*. **NeurIPS 2018**. arXiv:1808.09034.
- **[29]** Rainforth, T., Kosiorek, A. R., Le, T. A., Maddison, C. J., Igl, M., Wood, F. & Teh, Y. W. (2018). *Tighter variational bounds are not necessarily better*. **ICML 2018**. arXiv:1802.04537.
- **[30]** Alemi, A. A., Poole, B., Fischer, I., Dillon, J. V., Saurous, R. A. & Murphy, K. (2018). *Fixing a broken ELBO*. **ICML 2018**, PMLR 80.
- **[31]** Higgins, I., Matthey, L., Pal, A., Burgess, C., Glorot, X., Botvinick, M., Mohamed, S. & Lerchner, A. (2017). *β-VAE: Learning basic visual concepts with a constrained variational framework*. **ICLR 2017**.
- **[32]** Hoffman, M. D. & Johnson, M. J. (2016). *ELBO surgery: yet another way to carve up the variational evidence lower bound*. **NIPS Workshop on Advances in Approximate Bayesian Inference**.
- **[33]** Burgess, C. P., Higgins, I., Pal, A., Matthey, L., Watters, N., Desjardins, G. & Lerchner, A. (2018). *Understanding disentangling in β-VAE*. arXiv:1804.03599.

## Diagnóstico, extensiones y aplicaciones
- **[34]** Bowman, S. R., Vilnis, L., Vinyals, O., Dai, A., Jozefowicz, R. & Bengio, S. (2016). *Generating sentences from a continuous space*. **CoNLL 2016**. arXiv:1511.06349.
- **[35]** Kingma, D. P., Salimans, T., Jozefowicz, R., Chen, X., Sutskever, I. & Welling, M. (2016). *Improved variational inference with inverse autoregressive flow*. **NeurIPS 2016**. arXiv:1606.04934.
- **[36]** Lucas, J., Tucker, G., Grosse, R. & Norouzi, M. (2019). *Don't blame the ELBO! A linear VAE perspective on posterior collapse*. **NeurIPS 2019**. arXiv:1911.02469.
- **[37]** Dai, B. & Wipf, D. (2019). *Diagnosing and enhancing VAE models*. **ICLR 2019**. arXiv:1903.05789.
- **[38]** Sohn, K., Lee, H. & Yan, X. (2015). *Learning structured output representation using deep conditional generative models*. **NIPS 2015**.
- **[39]** Nalisnick, E., Matsukawa, A., Teh, Y. W., Gorur, D. & Lakshminarayanan, B. (2019). *Do deep generative models know what they don't know?* **ICLR 2019**. arXiv:1810.09136.

## Libros recomendados para profundizar
- Murphy, K. P. (2023). *Probabilistic Machine Learning: Advanced Topics*. MIT Press. (Cap. 21: VAEs.) https://probml.github.io/pml-book/book2.html
- Bishop, C. M. & Bishop, H. (2024). *Deep Learning: Foundations and Concepts*. Springer. (Caps. sobre variables latentes y VAEs.)
- Prince, S. J. D. (2023). *Understanding Deep Learning*. MIT Press. (Cap. 17: VAEs.) https://udlbook.github.io/udlbook/
- Chollet, F. (2021). *Deep Learning with Python*, 2nd ed. Manning. (Cap. 12: ejemplos VAE en Keras.)
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. (Cap. 10: Inferencia variacional.)